# Interactive Spectroscopy Analysis

A general-purpose tool for fitting peaks in Raman, fluorescence, and other
spectra — one file or many at once.

**Workflow:**
1. Load one or more spectrum files (`.txt`, `.csv`, `.dat` — two-column x/y, with optional header rows)
2. *(Optional)* Calibrate the wavenumber axis from Hg(Ar) / Kr lamp lines
3. Choose and tune the baseline subtraction
4. Pick a peak profile and adjust the detection threshold and the width / x-range gates
5. Read off the per-peak parameter table and the centre-vs-FWHM summary scatter
6. *(Optional)* Decompose a mixed dataset into end-member spectra (NMF)
7. Save the fitted plots and/or tabulated data

Run the code cells below, then use the interactive controls. A terse
**Quick reference** for every control is in the next cell.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display, clear_output
from itables import init_notebook_mode, show as itables_show
import os, glob, datetime, tempfile
%matplotlib inline

init_notebook_mode(all_interactive=False)

# Hard cap on the number of peaks fitted at once. Stops curve_fit from hanging
# when the user drags the threshold slider very low and find_peaks returns hundreds.
MAX_PEAKS = 30


# ==============================================================================
# Baseline estimation
# ==============================================================================

def arpls(y, lam=1e6, ratio=1e-3, max_iter=100):
    """Asymmetrically Reweighted Penalized Least Squares baseline."""
    N = len(y)
    D = sparse.diags([1, -2, 1], [0, 1, 2], shape=(N - 2, N), format='csc')
    H = lam * D.T @ D
    w = np.ones(N)
    for _ in range(max_iter):
        W = sparse.diags(w, format='csc')
        z = spsolve(W + H, w * y)
        d = y - z
        dn = d[d < 0]
        m, s = dn.mean(), dn.std()
        exponent = np.clip(2.0 * (d - (2.0 * s - m)) / s, -500, 500)
        w_new = 1.0 / (1.0 + np.exp(exponent))
        if np.linalg.norm(w_new - w) / np.linalg.norm(w) < ratio:
            break
        w = w_new
    return z


def poly_baseline(x, y, degree=5, n_iter=50):
    """Iterative polynomial baseline - removes points above the fit each iteration."""
    mask = np.ones(len(y), dtype=bool)
    for _ in range(n_iter):
        coeffs = np.polyfit(x[mask], y[mask], degree)
        bl = np.polyval(coeffs, x)
        new_mask = y <= bl
        if np.array_equal(new_mask, mask):
            break
        mask = new_mask
        if mask.sum() < degree + 1:
            break
    return np.polyval(np.polyfit(x[mask], y[mask], degree), x)


# ==============================================================================
# Peak profile functions
# ==============================================================================

def gaussian(x, amp, cen, fwhm):
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    return amp * np.exp(-((x - cen) ** 2) / (2 * sigma ** 2))


def lorentzian(x, amp, cen, fwhm):
    gamma = fwhm / 2
    return amp * gamma ** 2 / ((x - cen) ** 2 + gamma ** 2)


def pseudo_voigt(x, amp, cen, fwhm, eta):
    return amp * (eta * lorentzian(x, 1, cen, fwhm) +
                  (1 - eta) * gaussian(x, 1, cen, fwhm))


# Multi-peak models

# Vectorised across peaks: instead of a Python loop, stack the n_peaks parameter
# vectors and broadcast against x[:, None] so the entire (N, n_peaks) grid is
# evaluated in compiled numpy. Analytical Jacobians (jac_*) are supplied to
# curve_fit so it doesn't have to finite-difference the model ~(n_params+1)
# times per iteration -- a large speedup for 30-peak fits.

_FWHM_TO_SIGMA = 1.0 / (2.0 * np.sqrt(2.0 * np.log(2.0)))

def _gauss_grid(x, cens, fwhms):
    sig = fwhms * _FWHM_TO_SIGMA
    xc  = x[:, None] - cens                              # (N, n_peaks)
    return xc, sig, np.exp(-xc * xc / (2.0 * sig * sig))

def _lor_grid(x, cens, fwhms):
    gamma = fwhms / 2.0
    xc    = x[:, None] - cens
    d     = xc * xc + gamma * gamma
    return xc, gamma, d, gamma * gamma / d

def multi_gauss(x, *p):
    p = np.asarray(p, float)
    amps, cens, fwhms = p[0::3], p[1::3], p[2::3]
    _, _, G = _gauss_grid(x, cens, fwhms)
    return (amps * G).sum(axis=1)

def multi_lor(x, *p):
    p = np.asarray(p, float)
    amps, cens, fwhms = p[0::3], p[1::3], p[2::3]
    _, _, _, L = _lor_grid(x, cens, fwhms)
    return (amps * L).sum(axis=1)

def multi_pv(x, *p):
    p = np.asarray(p, float)
    amps, cens, fwhms, etas = p[0::4], p[1::4], p[2::4], p[3::4]
    _, _, G    = _gauss_grid(x, cens, fwhms)
    _, _, _, L = _lor_grid(x, cens, fwhms)
    return (amps * (etas * L + (1.0 - etas) * G)).sum(axis=1)

def multi_pv_shared(x, *p):
    """Pseudo-Voigt with one shared eta (last parameter)."""
    p = np.asarray(p, float)
    eta = p[-1]
    body = p[:-1]
    amps, cens, fwhms = body[0::3], body[1::3], body[2::3]
    _, _, G    = _gauss_grid(x, cens, fwhms)
    _, _, _, L = _lor_grid(x, cens, fwhms)
    return (amps * (eta * L + (1.0 - eta) * G)).sum(axis=1)


# Analytical Jacobians. Each returns an (N, n_params) array of partial
# derivatives, with columns interleaved in the same order as the parameter
# vector (amp_0, cen_0, fwhm_0[, eta_0], amp_1, ...).

def jac_gauss(x, *p):
    p = np.asarray(p, float)
    amps, cens, fwhms = p[0::3], p[1::3], p[2::3]
    xc, sig, G = _gauss_grid(x, cens, fwhms)
    g = amps * G
    J = np.empty((x.size, p.size))
    J[:, 0::3] = G                                    # dy/d(amp)
    J[:, 1::3] = g * xc / (sig * sig)                 # dy/d(cen)
    J[:, 2::3] = g * xc * xc / (sig * sig * fwhms)    # dy/d(fwhm)
    return J

def jac_lor(x, *p):
    p = np.asarray(p, float)
    amps, cens, fwhms = p[0::3], p[1::3], p[2::3]
    xc, gamma, d, L = _lor_grid(x, cens, fwhms)
    l = amps * L
    J = np.empty((x.size, p.size))
    J[:, 0::3] = L                                    # dy/d(amp)
    J[:, 1::3] = 2.0 * l * xc / d                     # dy/d(cen)
    J[:, 2::3] = l * xc * xc / (gamma * d)            # dy/d(fwhm)
    return J

def jac_pv(x, *p):
    p = np.asarray(p, float)
    amps, cens, fwhms, etas = p[0::4], p[1::4], p[2::4], p[3::4]
    xc, sig, G      = _gauss_grid(x, cens, fwhms)
    _,  gamma, d, L = _lor_grid(x, cens, fwhms)
    shape = etas * L + (1.0 - etas) * G
    dG_dcen  = G * xc / (sig * sig)
    dL_dcen  = 2.0 * L * xc / d
    dG_dfwhm = G * xc * xc / (sig * sig * fwhms)
    dL_dfwhm = L * xc * xc / (gamma * d)
    J = np.empty((x.size, p.size))
    J[:, 0::4] = shape
    J[:, 1::4] = amps * (etas * dL_dcen  + (1.0 - etas) * dG_dcen)
    J[:, 2::4] = amps * (etas * dL_dfwhm + (1.0 - etas) * dG_dfwhm)
    J[:, 3::4] = amps * (L - G)
    return J

def jac_pv_shared(x, *p):
    p = np.asarray(p, float)
    eta = p[-1]
    body = p[:-1]
    amps, cens, fwhms = body[0::3], body[1::3], body[2::3]
    xc, sig, G      = _gauss_grid(x, cens, fwhms)
    _,  gamma, d, L = _lor_grid(x, cens, fwhms)
    shape = eta * L + (1.0 - eta) * G
    dG_dcen  = G * xc / (sig * sig)
    dL_dcen  = 2.0 * L * xc / d
    dG_dfwhm = G * xc * xc / (sig * sig * fwhms)
    dL_dfwhm = L * xc * xc / (gamma * d)
    nbody = body.size
    J = np.empty((x.size, p.size))
    J[:, 0:nbody:3] = shape
    J[:, 1:nbody:3] = amps * (eta * dL_dcen  + (1.0 - eta) * dG_dcen)
    J[:, 2:nbody:3] = amps * (eta * dL_dfwhm + (1.0 - eta) * dG_dfwhm)
    J[:, nbody]     = (amps * (L - G)).sum(axis=1)
    return J


# ==============================================================================
# Evaluation helpers
# ==============================================================================

def eval_model(x, popt, profile):
    funcs = {'Gaussian': multi_gauss, 'Lorentzian': multi_lor,
             'Pseudo-Voigt': multi_pv, 'Pseudo-Voigt (shared \u03b7)': multi_pv_shared}
    return funcs[profile](x, *popt)


def eval_single_peak(x, popt, j, profile):
    if profile == 'Pseudo-Voigt (shared \u03b7)':
        eta = popt[-1]
        i = j * 3
        return pseudo_voigt(x, popt[i], popt[i+1], popt[i+2], eta)
    elif profile == 'Pseudo-Voigt':
        i = j * 4
        return pseudo_voigt(x, popt[i], popt[i+1], popt[i+2], popt[i+3])
    elif profile == 'Gaussian':
        i = j * 3
        return gaussian(x, popt[i], popt[i+1], popt[i+2])
    else:
        i = j * 3
        return lorentzian(x, popt[i], popt[i+1], popt[i+2])


def count_peaks(popt, profile):
    if popt is None:
        return 0
    if profile == 'Pseudo-Voigt (shared \u03b7)':
        return (len(popt) - 1) // 3
    stride = 4 if profile == 'Pseudo-Voigt' else 3
    return len(popt) // stride


# ==============================================================================
# Peak detection and fitting
# ==============================================================================

# Conservative noise multiplier: a peak must rise at least NOISE_K * sigma_noise
# above the local baseline to be detected, regardless of where the user has the
# slider. 5*sigma gives an expected false-positive rate well below 1 per spectrum
# for Gaussian noise on ~1000-point traces.
NOISE_K = 5.0

def estimate_noise_sigma(y):
    """Robust noise-sigma estimate via MAD of the first difference.

    The first difference of a constant signal + Gaussian noise has variance 2*sigma^2,
    so sigma_noise = MAD(diff) * 1.4826 / sqrt(2) (1.4826 is the Gaussian
    consistency factor; the median is robust to the few large diffs around any
    real peaks)."""
    if y.size < 3:
        return 0.0
    d = np.diff(y)
    return 1.4826 / np.sqrt(2.0) * np.median(np.abs(d - np.median(d)))

def masked_data_range(x, y, exclude_centres=None, exclude_window=8.0):
    """max-min of y with samples within +-exclude_window of any exclude centre
    removed. A bright excluded line (e.g. a calibration lamp peak) would
    otherwise dominate the data range and push the % height / amplitude
    thresholds up above the genuine, smaller Raman peaks."""
    y = np.asarray(y, float)
    if exclude_centres is not None and len(exclude_centres):
        xa = np.asarray(x, float)
        m = np.ones(len(xa), dtype=bool)
        for cx in exclude_centres:
            m &= np.abs(xa - cx) > exclude_window
        if m.any():
            y = y[m]
    return max(float(y.max() - y.min()), 1e-10)


def detect_peaks(x, y_sub, height_pct,
                 exclude_centres=None, exclude_window=3.0):
    """Find peaks above height_pct % of the data range, with a prominence filter
    AND a 5-sigma noise floor so a peakless / dark spectrum returns no peaks.

    If exclude_centres is given, any candidate within +/- exclude_window of any
    listed centre is dropped (used to remove calibration-lamp emission lines
    from the Raman peak set after wavenumber correction).

    Returns (idx, info) where info is a dict with diagnostic fields
    (sigma_noise, slider_threshold, applied_threshold, capped) so the UI can show
    what just happened."""
    # Threshold is a % of the Raman data range, with excluded (lamp-line)
    # regions masked out so a bright calibration peak can't inflate it.
    dr = masked_data_range(x, y_sub, exclude_centres, 8.0)
    h_slider = height_pct / 100.0 * dr
    sigma    = estimate_noise_sigma(y_sub)
    h_noise  = NOISE_K * sigma
    # Promote the looser of the two thresholds: never let the slider go below
    # the noise floor.
    h_abs    = max(h_slider, h_noise)
    idx, props = find_peaks(y_sub, height=h_abs, prominence=h_abs * 0.5, width=2)
    if exclude_centres is not None and len(idx):
        x_peaks = x[idx]
        keep = np.ones(len(idx), dtype=bool)
        for cx in exclude_centres:
            keep &= np.abs(x_peaks - cx) > exclude_window
        idx = idx[keep]
        for k in props:
            props[k] = props[k][keep]
    capped = len(idx) > MAX_PEAKS
    if capped:
        order = np.argsort(props['peak_heights'])[::-1][:MAX_PEAKS]
        idx = np.sort(idx[order])
    info = dict(sigma_noise=sigma, slider_threshold=h_slider,
                applied_threshold=h_abs, capped=capped, data_range=dr,
                noise_floor_wins=(h_noise > h_slider))
    return idx, info


def snap_to_local_max(x, y, ref_centres, search_pts=10):
    """For each reference peak centre, return the index of the nearest local
    maximum in (x, y) within ±search_pts samples of where the reference centre
    falls on the new x-axis.

    Used so the multi-file fit can adopt the reference's peak *topology*
    (count, approximate positions) while grounding each spectrum's actual
    initial amplitudes/centres in its own data. Without this, every spectrum
    would inherit the reference's amplitude bounds, which fails badly when the
    reference is much weaker or noisier than the other spectra."""
    out = []
    n = len(x)
    for xc in ref_centres:
        j0 = int(np.argmin(np.abs(x - xc)))
        lo = max(0, j0 - search_pts)
        hi = min(n - 1, j0 + search_pts)
        out.append(lo + int(np.argmax(y[lo:hi + 1])))
    return np.asarray(out, dtype=int)


def union_peak_centres(per_file_items, merge_tol, max_peaks):
    """Merge peak centres detected in multiple files into a single sorted list.

    per_file_items: list of lists of (centre, height) tuples, one inner list
        per file. centres are in absolute x-units.
    merge_tol: peaks closer than this are treated as the same physical peak
        (single-linkage clustering by centre proximity).
    max_peaks: hard cap on the number of union centres returned. When the
        cluster count exceeds this, clusters are ranked first by how many
        distinct files contributed to them, then by mean detected height; the
        top max_peaks are kept.

    Returns a sorted 1D numpy array of union centres (cluster medians)."""
    items = []  # (centre, height, file_idx)
    for fi, ch_list in enumerate(per_file_items):
        for c, h in ch_list:
            items.append((float(c), float(h), fi))
    items.sort(key=lambda t: t[0])
    if not items:
        return np.array([])
    clusters = [[items[0]]]
    for it in items[1:]:
        if it[0] - clusters[-1][-1][0] <= merge_tol:
            clusters[-1].append(it)
        else:
            clusters.append([it])
    if len(clusters) > max_peaks:
        def rank(cl):
            n_files = len(set(it[2] for it in cl))
            mean_h  = float(np.mean([it[1] for it in cl]))
            return (-n_files, -mean_h)
        clusters = sorted(clusters, key=rank)[:max_peaks]
    centres = [float(np.median([it[0] for it in cl])) for cl in clusters]
    return np.asarray(sorted(centres))


def build_initial_guess(x, y_sub, peak_idx, profile, dx):
    """Construct p0 / lower / upper bounds, the multi-peak model, and its analytical Jacobian."""
    p0, lo, hi = [], [], []
    if profile in ('Gaussian', 'Lorentzian'):
        for i in peak_idx:
            p0 += [y_sub[i], x[i], 5*dx]
            lo += [0, x[i]-15*dx, 0.5*dx]
            hi += [y_sub[i]*5, x[i]+15*dx, 150*dx]
        if profile == 'Gaussian':
            func, jac = multi_gauss, jac_gauss
        else:
            func, jac = multi_lor, jac_lor
    elif profile == 'Pseudo-Voigt':
        for i in peak_idx:
            p0 += [y_sub[i], x[i], 5*dx, 0.5]
            lo += [0, x[i]-15*dx, 0.5*dx, 0.0]
            hi += [y_sub[i]*5, x[i]+15*dx, 150*dx, 1.0]
        func, jac = multi_pv, jac_pv
    else:
        for i in peak_idx:
            p0 += [y_sub[i], x[i], 5*dx]
            lo += [0, x[i]-15*dx, 0.5*dx]
            hi += [y_sub[i]*5, x[i]+15*dx, 150*dx]
        p0.append(0.5); lo.append(0.0); hi.append(1.0)
        func, jac = multi_pv_shared, jac_pv_shared
    return p0, lo, hi, func, jac


def fit_spectrum(x, y_sub, p0, lo, hi, func, jac=None, fit_mask=None):
    """Run curve_fit with an optional analytical Jacobian.

    fit_mask (bool array over x) selects the points actually fitted -- used to
    drop calibration-lamp regions, whose huge unmodelled residuals would
    otherwise dominate the least-squares cost and pull broad spurious peaks
    into the gaps between bright lamp lines. The returned fit_curve is always
    evaluated on the FULL x so plotting/residuals are unaffected.

    Returns (popt, perr, fit_curve); (None, None, zeros) on failure."""
    xf = x if fit_mask is None else x[fit_mask]
    yf = y_sub if fit_mask is None else y_sub[fit_mask]
    try:
        popt, pcov = curve_fit(func, xf, yf, p0=p0, bounds=(lo, hi),
                               jac=jac, x_scale='jac', maxfev=30000)
        perr = np.sqrt(np.maximum(np.diag(pcov), 0.0))
        return popt, perr, func(x, *popt)
    except (RuntimeError, ValueError):
        return None, None, np.zeros_like(x)


def peak_params_with_errors(popt, perr, profile, n_peaks):
    """One dict per peak: cen, cen_err, fwhm, fwhm_err, amp, amp_err (+ eta for full PV)."""
    if popt is None or perr is None:
        return [None] * n_peaks
    rows = []
    if profile == 'Pseudo-Voigt':
        for j in range(n_peaks):
            i = j * 4
            rows.append(dict(
                amp=popt[i], amp_err=perr[i],
                cen=popt[i+1], cen_err=perr[i+1],
                fwhm=abs(popt[i+2]), fwhm_err=perr[i+2],
                eta=popt[i+3], eta_err=perr[i+3]))
    else:
        # Gaussian, Lorentzian and PV-shared all have stride 3 for the per-peak block
        for j in range(n_peaks):
            i = j * 3
            rows.append(dict(
                amp=popt[i], amp_err=perr[i],
                cen=popt[i+1], cen_err=perr[i+1],
                fwhm=abs(popt[i+2]), fwhm_err=perr[i+2]))
    return rows


def build_single_file_df(fd, profile):
    """Tidy DataFrame for one spectrum: rows = peaks that pass that spectrum's
    own threshold gates (peak_kept), columns = parameters with errors. Without
    the peak_kept filter the table would list every union peak even when most
    of them don't appear in this spectrum, which is confusing when only one
    file is visible. Returns None if no popt, or an empty DataFrame if no
    peaks survived the gates."""
    popt = fd.get('popt'); perr = fd.get('perr')
    if popt is None or perr is None:
        return None
    n    = count_peaks(popt, profile)
    kept = fd.get('peak_kept', [True] * n)
    cal  = fd.get('cal_info')
    # Per-peak systematic calibration uncertainty (0 when cal is off). Kept as
    # a SEPARATE column from the random fit error rather than summed in, since
    # it's common-mode across peaks in one spectrum (cancels in same-spectrum
    # peak differences, but adds when comparing across spectra).
    def _cerr(centre):
        return float(cal_position_sigma(cal, float(centre))) if cal else 0.0
    # 'Centre err (total)' = quadrature sum sqrt(fit^2 + cal^2): the overall
    # uncertainty to quote for an absolute position or a cross-spectrum
    # comparison. Within one spectrum the cal term is common-mode and cancels,
    # so use 'Centre err (fit)' there instead.
    if profile == 'Pseudo-Voigt':
        cols = ['Centre', 'Centre err (fit)', 'Centre err (cal)',
                'Centre err (total)',
                'FWHM', 'FWHM err', 'Amplitude', 'Amplitude err',
                '\u03b7', '\u03b7 err']
        def _row(j):
            i = j * 4
            ce = _cerr(popt[i+1])
            return [popt[i+1], perr[i+1], ce, float(np.hypot(perr[i+1], ce)),
                    abs(popt[i+2]), perr[i+2], popt[i], perr[i],
                    popt[i+3], perr[i+3]]
    else:
        cols = ['Centre', 'Centre err (fit)', 'Centre err (cal)',
                'Centre err (total)',
                'FWHM', 'FWHM err', 'Amplitude', 'Amplitude err']
        def _row(j):
            i = j * 3
            ce = _cerr(popt[i+1])
            return [popt[i+1], perr[i+1], ce, float(np.hypot(perr[i+1], ce)),
                    abs(popt[i+2]), perr[i+2], popt[i], perr[i]]
    data, index = [], []
    for j in range(n):
        if j < len(kept) and not kept[j]:
            continue
        data.append(_row(j))
        index.append(f'Peak {j+1}')
    df = pd.DataFrame(data, columns=cols, index=index)
    df.index.name = 'peak'
    return df


def build_multi_file_df(files, profile, n_peaks, useful_mask=None):
    """DataFrame for many spectra: rows = files, columns = MultiIndex (peak, metric).
    Cells are NaN where the peak fell below the threshold/width gate for that
    spectrum. If useful_mask is given, only union peaks that survived in at
    least one file are included as columns (so all-NaN columns are dropped
    while the surviving peak numbers stay stable, e.g. 1, 2, 4, ...)."""
    if useful_mask is None:
        useful_mask = [True] * n_peaks
    used = [j for j in range(n_peaks) if j < len(useful_mask) and useful_mask[j]]
    # Column labels are SEQUENTIAL among useful peaks (so '...Peak 5...' isn't
    # rendered next to four guide-lines on the plot). Mapping `popt_idx -> j`
    # is preserved through `used` for indexing the parameter arrays.
    metrics = ['centre', 'centre err (fit)', 'centre err (cal)',
               'centre err (total)', 'FWHM', 'FWHM err']
    cols = pd.MultiIndex.from_product(
        [[f'Peak {disp+1}' for disp in range(len(used))], metrics],
        names=['peak', 'metric'])
    rows, index = [], []
    for fd in files:
        index.append(fd['name'])
        params = peak_params_with_errors(
            fd.get('popt'), fd.get('perr'), profile, n_peaks)
        kept = fd.get('peak_kept', [True] * n_peaks)
        cal = fd.get('cal_info')
        row = []
        for j in used:
            p = params[j] if j < len(params) else None
            if p is None or (j < len(kept) and not kept[j]):
                row += [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]
            else:
                cerr = (float(cal_position_sigma(cal, p['cen']))
                        if cal else 0.0)
                row += [p['cen'], p['cen_err'], cerr,
                        float(np.hypot(p['cen_err'], cerr)),
                        p['fwhm'], p['fwhm_err']]
        rows.append(row)
    df = pd.DataFrame(rows, index=index, columns=cols)
    df.index.name = 'file'
    return df





# ==============================================================================
# Wavenumber calibration (optional)
#
# When calibration lamps are on during acquisition, their emission lines appear
# as sharp features in every spectrum. fit_calibration_lines locates each
# expected line in a window of +/- a few cm-1 around its literature position,
# fits a single Gaussian for sub-sample precision, and returns the measured
# positions. compute_wavenumber_correction then linear-fits (measured - known)
# against known wavenumber (same model as the original TREND-based spreadsheet),
# giving a per-spectrum correction that can be applied to x_raw before all
# downstream processing.
# ==============================================================================

def parse_cal_lines(text):
    """Parse a calibration-line spec block.

    Each non-empty, non-comment line is "<label> <wavenumber> [wavelength_nm]".
    Leading non-numeric tokens form the label; the first numeric is the
    wavenumber (cm^-1, the value used for the correction) and an optional
    second numeric is the air wavelength in nm (informational only). Lines
    starting with '#' are ignored.

    Returns a list of (label, wavenumber, wavelength) tuples; wavelength is
    None when not given."""
    out = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s or s.startswith('#'):
            continue
        label_toks, nums = [], []
        for t in s.split():
            try:
                nums.append(float(t))
            except ValueError:
                if not nums:          # only leading non-numeric tokens label
                    label_toks.append(t)
        if not nums:
            continue
        label = ' '.join(label_toks)
        wn = nums[0]
        wl = nums[1] if len(nums) > 1 else None
        out.append((label, wn, wl))
    return out


def parse_manual_peaks(text):
    """Parse extra peak centres (cm^-1) the user wants force-fitted on top of
    the auto-detected ones -- for shoulders / blended peaks that aren't local
    maxima and so can't be found by find_peaks. One or more numbers per line
    (space/comma separated); '#' starts a comment."""
    out = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s or s.startswith('#'):
            continue
        for tok in s.replace(',', ' ').split():
            try:
                out.append(float(tok))
            except ValueError:
                pass
    return out


# Minimum SNR (fitted-Gaussian amplitude / global noise sigma) for a
# calibration-line fit to be considered "found". 10 leaves a clear gap above
# the typical max-of-N-samples noise spike (which rarely exceeds ~6 sigma in a
# small window) while staying well below the SNR (50-100) seen for real
# calibration lamp lines, which are amongst the brightest features in the
# spectrum by design.
CAL_SNR_MIN = 10.0

# Half-width (cm^-1) of the region masked around each calibration-lamp line.
# Used BOTH to drop lamp regions from the least-squares fit AND to exclude
# them from Raman peak detection -- the two must match. If detection used a
# narrower window than the fit mask, a feature detected in the gap (outside
# the detection window, inside the fit mask) would enter the peak union and
# then be fitted against points that have all been masked away, giving a
# spurious peak with a wildly batch-/window-dependent amplitude.
LAMP_MASK_HW = 8.0

def fit_calibration_lines(x, y, expected_lines, window, snr_min=CAL_SNR_MIN):
    """For each expected (label, known_x), locate a sharp peak within
    [known_x - window, known_x + window] in (x, y) and fit a single Gaussian
    for sub-sample-precision centre.

    A fit is only accepted if its amplitude (above the windowed linear
    baseline) is at least snr_min times the global noise sigma of the spectrum
    (estimate_noise_sigma over the full y, which is robust to real peaks and
    much more stable than estimating sigma from the ~10 samples inside a
    single search window). This rejects the failure mode where one of the
    expected lines is simply not present in the data and curve_fit latches
    onto a random noise spike.

    Returns a list of dicts (one per expected line) with keys:
        label, known, measured, fwhm, snr, success, error
    where 'measured' is None if no real peak was found."""
    sigma_global = estimate_noise_sigma(np.asarray(y, float))
    out = []
    for _item in expected_lines:
        label, kx = _item[0], _item[1]
        wl = _item[2] if len(_item) > 2 else None
        mask = (x >= kx - window) & (x <= kx + window)
        n_in = int(mask.sum())
        if n_in < 5:
            out.append(dict(label=label, known=kx, wavelength=wl, measured=None,
                            fwhm=None, cen_err=None, snr=None, success=False,
                            error=f'only {n_in} samples in window'))
            continue
        x_w = x[mask].astype(float)
        y_w = y[mask].astype(float)
        # Subtract a straight line through the window endpoints so the
        # Gaussian fit isn't biased by sloping background.
        bl = np.linspace(y_w[0], y_w[-1], n_in)
        y_d = y_w - bl
        j = int(np.argmax(y_d))
        amp_init  = max(float(y_d[j]), 1e-6)
        cen_init  = float(x_w[j])
        dx        = float(np.median(np.diff(x_w))) if n_in > 1 else 1.0
        fwhm_init = max(3.0 * dx, 1e-6)
        try:
            popt, pcov = curve_fit(
                gaussian, x_w, y_d,
                p0=[amp_init, cen_init, fwhm_init],
                bounds=([0.0,            kx - window, 0.5 * dx],
                        [amp_init * 10 + 1e-3, kx + window, window]),
                maxfev=5000)
        except (RuntimeError, ValueError) as e:
            out.append(dict(label=label, known=kx, wavelength=wl, measured=None,
                            fwhm=None, cen_err=None, snr=None, success=False,
                            error=str(e)))
            continue
        amp_fit  = float(popt[0])
        cen_fit  = float(popt[1])
        fwhm_fit = abs(float(popt[2]))
        cen_err  = float(np.sqrt(abs(pcov[1, 1]))) if np.all(np.isfinite(pcov)) else np.nan
        # MAD-based noise sigma estimated from the DATA in the window, not the
        # residual. The residual under-estimates the noise: if curve_fit
        # captured a single high noise sample with a narrow Gaussian, the
        # residual at that sample is ~0 and the rest is small noise, so MAD
        # would shrink and SNR would look artificially huge. MAD is robust to
        # a handful of peak samples, so the noise floor estimate is still
        # correct when a real peak is in the window.
        # SNR against the GLOBAL noise sigma. Using an in-window MAD is
        # tempting but ~10 samples is too few for a stable estimate -- it
        # sometimes lands artificially low and lets noise spikes through.
        # The global estimate (MAD of diff(y) over the whole spectrum) is
        # robust to real peaks and uses hundreds of samples.
        snr = amp_fit / max(sigma_global, 1e-6)
        if snr < snr_min:
            out.append(dict(label=label, known=kx, wavelength=wl, measured=None,
                            fwhm=fwhm_fit, cen_err=cen_err, snr=snr,
                            success=False,
                            error=f'no peak above noise (SNR={snr:.1f} '
                                  f'< {snr_min:.1f})'))
            continue
        out.append(dict(label=label, known=kx, wavelength=wl, measured=cen_fit,
                        fwhm=fwhm_fit, cen_err=cen_err, snr=snr,
                        success=True, error=None))
    return out


def cal_line_note(d):
    """Short human note on why a calibration line was or was not used, for
    the diagnostic table / status line. Returns (text, is_rejected).
    A line can be dropped three ways: too few samples in its search window
    (line near the spectrum edge or off by more than the window), curve_fit
    failing, or its fitted amplitude falling below CAL_SNR_MIN x noise."""
    if d['success']:
        snr = d.get('snr')
        return (f'SNR {snr:.0f}' if snr is not None else 'used'), False
    snr = d.get('snr'); err = d.get('error') or ''
    if snr is not None:
        return f'SNR {snr:.1f} < {CAL_SNR_MIN:.0f}', True
    if 'samples in window' in err:
        return 'no data in \u00b1window', True
    return 'fit failed', True


def _polyfit_diff(knowns, diffs, o):
    """Fit diff = polyval(coeffs, known) at order o (o=0 is a constant offset)."""
    if o == 0:
        return np.array([float(np.mean(diffs))])
    return np.polyfit(knowns, diffs, o)


def _order_statistics(knowns, diffs, cen_err, n):
    """Per-order model-selection statistics for choosing the fit order.

    RMS always falls with order (useless for selection). The penalised
    criteria below turn back up when extra terms stop earning their keep:
      - chi2_red: weighted SSR / dof; ~1 means 'fits to within the line
        position errors', <1 means overfitting (fitting noise).
      - AICc: small-sample-corrected Akaike; undefined (inf) once params
        reach n-1, i.e. it refuses orders the data can't support.
      - loocv: leave-one-out CV RMS; rises when the fit stops generalising.
    Returns {order: {rms, chi2_red, aicc, loocv, p, dof}}."""
    ce = np.where(np.isfinite(cen_err) & (cen_err > 0), cen_err, np.nan)
    ce = np.full(n, 0.1) if np.all(np.isnan(ce)) else np.where(
        np.isnan(ce), np.nanmedian(ce), ce)
    ce = np.maximum(ce, 1e-3)
    stats = {}
    for o in range(0, min(3, n - 1) + 1):
        p = o + 1
        c = _polyfit_diff(knowns, diffs, o)
        r = diffs - np.polyval(c, knowns)
        ssr = float(np.sum(r ** 2))
        chi2 = float(np.sum((r / ce) ** 2))
        dof = n - p
        aic = chi2 + 2 * p
        aicc = aic + (2 * p * (p + 1) / (n - p - 1) if (n - p - 1) > 0 else np.inf)
        if n - 1 >= p:
            errs = []
            for i in range(n):
                k = np.arange(n) != i
                ci = _polyfit_diff(knowns[k], diffs[k], o)
                errs.append(diffs[i] - np.polyval(ci, knowns[i]))
            loocv = float(np.sqrt(np.mean(np.square(errs))))
        else:
            loocv = float('nan')
        stats[o] = dict(rms=float(np.sqrt(ssr / n)),
                        chi2_red=(chi2 / dof if dof > 0 else float('inf')),
                        aicc=aicc, loocv=loocv, p=p, dof=dof)
    return stats, ce


def compute_wavenumber_correction(found, allow_offset=False, order='auto'):
    """Fit a polynomial wavenumber correction (diff = polyval(coeffs, known),
    diff = measured - known) and return everything needed to apply it, judge
    it, and propagate its uncertainty.

    order: 'auto' (1 line->offset, 2-3->linear, 4+->quadratic), 'aicc' (pick
    the order minimising AICc -- the statistically-justified choice for few
    noisy lines), or an explicit int 0..3. Clamped to feasible + capped at
    cubic. 1 line with allow_offset=False is declined.

    Returns None if no correction is possible, else a dict with keys:
      coeffs, order, rms, residuals, n_used,
      x0, s, invVtV, sigma   -- machinery for the position-uncertainty band
                                sigma_cal(x) = sigma * sqrt(v(x) invVtV v(x))
                                in the centred/scaled coord u=(x-x0)/s,
      stats                  -- per-order model-selection statistics,
      knowns, measured, cen_err."""
    pairs = [(d['known'], d['measured'], d.get('cen_err')) for d in found
             if d['success']]
    n = len(pairs)
    if n == 0:
        return None
    knowns = np.array([p[0] for p in pairs], float)
    meas   = np.array([p[1] for p in pairs], float)
    cen_err = np.array([p[2] if p[2] is not None else np.nan for p in pairs], float)
    diffs  = meas - knowns
    stats, ce = _order_statistics(knowns, diffs, cen_err, n)

    if order == 'auto':
        o = 0 if n == 1 else (1 if n <= 3 else 2)
    elif order == 'aicc':
        finite = {oo: st['aicc'] for oo, st in stats.items()
                  if np.isfinite(st['aicc'])}
        o = min(finite, key=finite.get) if finite else min(1, n - 1)
    else:
        o = int(order)
    o = max(0, min(o, 3, n - 1))
    if o == 0 and n == 1 and not allow_offset:
        return None

    coeffs = _polyfit_diff(knowns, diffs, o)
    resid  = diffs - np.polyval(coeffs, knowns)
    rms    = float(np.sqrt(np.mean(resid ** 2)))

    # Covariance machinery for the position-uncertainty band, in a
    # centred/scaled coordinate for conditioning.
    x0 = float(knowns.mean()); s = float(max(knowns.std(), 1.0))
    V = np.vander((knowns - x0) / s, o + 1)
    invVtV = np.linalg.pinv(V.T @ V)
    sigma = float(np.sqrt(np.mean(ce ** 2)))
    sigma = max(sigma, 0.02)
    return dict(coeffs=coeffs, order=o, rms=rms, residuals=resid, n_used=n,
                x0=x0, s=s, invVtV=invVtV, sigma=sigma, stats=stats,
                knowns=knowns, measured=meas, cen_err=ce)


def cal_position_sigma(cal, x):
    """1-sigma calibration uncertainty (cm^-1) at wavenumber(s) x, from the
    covariance stored in a compute_wavenumber_correction result. This is the
    systematic uncertainty the x-axis correction adds to every fitted peak
    position. Returns 0 for x where cal is None."""
    scalar = np.ndim(x) == 0
    xa = np.atleast_1d(np.asarray(x, float))
    if cal is None:
        out = np.zeros_like(xa)
    else:
        v = np.vander((xa - cal['x0']) / cal['s'], cal['order'] + 1)
        var = np.einsum('ij,jk,ik->i', v, cal['invVtV'], v)
        out = cal['sigma'] * np.sqrt(np.clip(var, 0.0, None))
    return float(out[0]) if scalar else out


def apply_wavenumber_correction(x_raw, coeffs):
    """Apply the polynomial correction from compute_wavenumber_correction.
    x_corrected = x_raw - polyval(coeffs, x_raw)."""
    return x_raw - np.polyval(np.asarray(coeffs, float), x_raw)


# ==============================================================================
# Spectral decomposition (multivariate curve resolution)
#
# Given many spectra that share the same few underlying components in
# different proportions, factor them into a small set of end-member spectra
# (H) and per-spectrum abundance weights (W). Uses NMF (non-negative both
# spectra and weights), which is physically appropriate for Raman intensities
# and mixture fractions.
# ==============================================================================

def build_decomp_matrix(spectra):
    """Stack a list of fd dicts into a non-negative (n_spectra, n_x) matrix on
    a common x grid. Returns (x_common, Y, names) or (None, None, []) if not
    enough usable data. Uses the intersection of x-axes (so every spectrum
    actually contributes over the whole window) and linearly interpolates each
    spectrum onto the common grid."""
    valid = [fd for fd in spectra
             if fd.get('x_trim') is not None and fd.get('y_sub') is not None
             and len(fd['x_trim']) > 1]
    if len(valid) < 2:
        return None, None, []
    x_lo = max(float(fd['x_trim'][0])  for fd in valid)
    x_hi = min(float(fd['x_trim'][-1]) for fd in valid)
    if x_hi <= x_lo:
        return None, None, []
    dx_min = min(float(np.median(np.diff(fd['x_trim']))) for fd in valid)
    n_pts = max(int((x_hi - x_lo) / dx_min) + 1, 100)
    x_common = np.linspace(x_lo, x_hi, n_pts)
    Y = np.zeros((len(valid), n_pts))
    for i, fd in enumerate(valid):
        Y[i] = np.interp(x_common, fd['x_trim'], fd['y_sub'])
    Y = np.clip(Y, 0.0, None)   # NMF rejects negatives
    names = [fd['name'] for fd in valid]
    return x_common, Y, names


def decompose_nmf(Y, n_components, normalize=True, random_state=0):
    """Run scikit-learn NMF on Y (n_spectra, n_wavenumbers).

    If normalize=True, each row of Y is scaled to unit area before fitting so
    absolute brightness doesn't dominate the decomposition. After fitting, the
    end-member spectra (rows of H) are rescaled to unit area and the
    normalisation absorbed into W, so each W[i,k] is the share of spectrum i
    accounted for by component k.

    Returns (H, W, r2):
        H : (n_components, n_wavenumbers) end-member spectra (unit area each)
        W : (n_spectra,   n_components)   per-spectrum component weights
        r2: reconstruction R^2 in the (possibly normalised) fitting frame."""
    from sklearn.decomposition import NMF  # local import keeps the notebook
                                            # usable without sklearn until you
                                            # actually invoke decomposition.
    X = np.asarray(Y, float)
    if normalize:
        norms = X.sum(axis=1, keepdims=True)
        X_fit = X / np.maximum(norms, 1e-10)
    else:
        X_fit = X
    n_components = max(2, min(int(n_components), min(X_fit.shape) - 1))
    model = NMF(n_components=n_components, init='nndsvda',
                max_iter=2000, tol=1e-5, random_state=random_state)
    W = model.fit_transform(X_fit)
    H = model.components_
    # Rescale each end-member row to unit area; push the scale into W.
    row_sums = H.sum(axis=1, keepdims=True)
    H = H / np.maximum(row_sums, 1e-10)
    W = W * row_sums.T
    # Reconstruction R^2
    Y_pred = W @ H
    ss_res = float(((X_fit - Y_pred) ** 2).sum())
    ss_tot = float(((X_fit - X_fit.mean()) ** 2).sum()) + 1e-30
    r2 = 1.0 - ss_res / ss_tot
    return H, W, r2


def fit_endmembers_to_peaks(x_common, H, peak_centres, profile):
    """Fit each end-member spectrum (row of H) to a known set of peak
    centres, using the same union peak set the per-spectrum Raman fit uses.

    Returns a list of dicts (one per component) with keys popt, perr, n_peaks
    where popt/perr are the curve_fit outputs and n_peaks = len(peak_centres).
    Failed components have popt=None, perr=None."""
    if len(peak_centres) == 0:
        return [dict(popt=None, perr=None, n_peaks=0) for _ in range(H.shape[0])]
    dx = float(np.median(np.diff(x_common)))
    results = []
    for k in range(H.shape[0]):
        y_sub = np.asarray(H[k], float)
        idx = snap_to_local_max(x_common, y_sub, peak_centres, search_pts=10)
        p0, lo, hi, func, jac = build_initial_guess(
            x_common, y_sub, idx, profile, dx)
        popt, perr, _ = fit_spectrum(x_common, y_sub, p0, lo, hi, func, jac)
        results.append(dict(popt=popt, perr=perr, n_peaks=len(peak_centres)))
    return results


def build_endmember_df(fit_results, profile, peak_indices):
    """DataFrame of fitted peak parameters per end-member component.

    Rows are 'Component 1', 'Component 2', ...; columns are a MultiIndex of
    (peak label, metric) where peak labels are formatted from peak_indices
    so they line up with the Raman peak table elsewhere (e.g. Peak 1, Peak 2,
    Peak 4, ... if Peak 3 was dropped as universally blanked)."""
    # Labels are SEQUENTIAL among the peaks we have (so the column header
    # matches the on-plot guide line number). peak_indices still maps each
    # column slot back to the original popt index for cross-reference.
    metrics = ['centre', 'centre err', 'FWHM', 'FWHM err',
               'amplitude', 'amp err']
    cols = pd.MultiIndex.from_product(
        [[f'Peak {disp+1}' for disp in range(len(peak_indices))], metrics],
        names=['peak', 'metric'])
    rows, index = [], []
    for k, res in enumerate(fit_results):
        index.append(f'Component {k+1}')
        params = peak_params_with_errors(
            res['popt'], res['perr'], profile, res['n_peaks'])
        row = []
        for slot, j in enumerate(peak_indices):
            p = params[slot] if slot < len(params) else None
            if p is None:
                row += [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]
            else:
                row += [p['cen'], p['cen_err'],
                        p['fwhm'], p['fwhm_err'],
                        p.get('amp', np.nan), p.get('amp_err', np.nan)]
        rows.append(row)
    df = pd.DataFrame(rows, index=index, columns=cols)
    df.index.name = 'component'
    return df


# ==============================================================================
# File I/O
# ==============================================================================

def auto_load_spectrum(filepath):
    """Auto-detect header/footer and load a two-column spectrum."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    skip = 0
    for i, line in enumerate(lines):
        parts = line.strip().replace(',', ' ').split()
        if len(parts) >= 2:
            try:
                float(parts[0]); float(parts[1])
                skip = i; break
            except ValueError:
                continue

    n_rows = 0
    for i in range(skip, len(lines)):
        parts = lines[i].strip().replace(',', ' ').split()
        if len(parts) >= 2:
            try:
                float(parts[0]); float(parts[1])
                n_rows += 1
            except ValueError:
                break
        else:
            break

    for delim in ['\t', ',', None]:
        try:
            data = np.loadtxt(filepath, skiprows=skip, max_rows=n_rows, delimiter=delim)
            if data.ndim == 2 and data.shape[1] >= 2:
                return data[:, 0], data[:, 1], skip, n_rows
        except Exception:
            continue
    raise ValueError(f"Could not parse {filepath}")


def save_fit_results(filepath, x, y_raw, baseline, y_sub, fit_curve, residual,
                     popt, profile, algo, algo_param, filename):
    """Write a self-documenting text file with all fit results for one spectrum."""
    with open(filepath, 'w') as f:
        f.write(f"# Spectroscopy fit results\n")
        f.write(f"# Source: {filename}\n")
        f.write(f"# Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        if algo == 'arPLS':
            f.write(f"# Baseline: arPLS (lambda = {algo_param:.2e})\n")
        else:
            f.write(f"# Baseline: Polynomial (degree = {int(algo_param)})\n")
        f.write(f"# Profile: {profile}\n#\n")

        if popt is not None:
            n = count_peaks(popt, profile)
            f.write(f"# Fitted peaks: {n}\n")
            if profile == 'Pseudo-Voigt (shared \u03b7)':
                f.write(f"# Shared eta = {popt[-1]:.4f}\n")
                f.write(f"# {'Peak':>4s}  {'Centre':>12s}  {'Amplitude':>12s}  {'FWHM':>12s}\n")
                for j in range(n):
                    i = j * 3
                    f.write(f"# {j+1:4d}  {popt[i+1]:12.4f}  {popt[i]:12.4f}  {abs(popt[i+2]):12.4f}\n")
            elif profile == 'Pseudo-Voigt':
                f.write(f"# {'Peak':>4s}  {'Centre':>12s}  {'Amplitude':>12s}  {'FWHM':>12s}  {'eta':>8s}\n")
                for j in range(n):
                    i = j * 4
                    f.write(f"# {j+1:4d}  {popt[i+1]:12.4f}  {popt[i]:12.4f}  {abs(popt[i+2]):12.4f}  {popt[i+3]:8.4f}\n")
            else:
                f.write(f"# {'Peak':>4s}  {'Centre':>12s}  {'Amplitude':>12s}  {'FWHM':>12s}\n")
                for j in range(n):
                    i = j * 3
                    f.write(f"# {j+1:4d}  {popt[i+1]:12.4f}  {popt[i]:12.4f}  {abs(popt[i+2]):12.4f}\n")
        f.write(f"#\n# Columns: x  raw_intensity  baseline  subtracted  fit  residual\n")
        for i in range(len(x)):
            f.write(f"{x[i]:.6f}\t{y_raw[i]:.6f}\t{baseline[i]:.6f}\t"
                    f"{y_sub[i]:.6f}\t{fit_curve[i]:.6f}\t{residual[i]:.6f}\n")


def save_results_table(filepath, files, profile, n_peaks):
    """Write a TSV summary: one row per file, columns for each peak (centre/FWHM/amp ± err).
    Cells are blank where the fitted peak fell below the height threshold for that spectrum."""
    with open(filepath, 'w') as f:
        f.write(f"# Multi-spectrum fit results\n")
        f.write(f"# Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"# Profile: {profile}\n")
        f.write(f"# Number of peaks: {n_peaks}\n")
        f.write(f"# Blank cells indicate the peak fell below the height threshold for that spectrum.\n")
        f.write(f"# Centre errors: _fit = random (fit); _cal = systematic (wavenumber calibration, common-mode within a spectrum); _total = sqrt(fit^2 + cal^2).\n#\n")
        cols = ['filename']
        for j in range(n_peaks):
            cols += [f'peak{j+1}_centre',
                     f'peak{j+1}_centre_err_fit',
                     f'peak{j+1}_centre_err_cal',
                     f'peak{j+1}_centre_err_total',
                     f'peak{j+1}_fwhm',   f'peak{j+1}_fwhm_err',
                     f'peak{j+1}_amp',    f'peak{j+1}_amp_err']
        f.write('\t'.join(cols) + '\n')
        for fd in files:
            row = [fd['name']]
            params = peak_params_with_errors(fd.get('popt'), fd.get('perr'), profile, n_peaks)
            kept = fd.get('peak_kept', [True] * n_peaks)
            for j, p in enumerate(params):
                if p is None or (j < len(kept) and not kept[j]):
                    row += ['', '', '', '', '', '', '', '']
                else:
                    cerr = (float(cal_position_sigma(fd.get('cal_info'), p['cen']))
                            if fd.get('cal_info') else 0.0)
                    row += [f"{p['cen']:.6f}",  f"{p['cen_err']:.6f}",
                            f"{cerr:.6f}", f"{float(np.hypot(p['cen_err'], cerr)):.6f}",
                            f"{p['fwhm']:.6f}", f"{p['fwhm_err']:.6f}",
                            f"{p['amp']:.6f}",  f"{p['amp_err']:.6f}"]
            f.write('\t'.join(row) + '\n')


## Quick reference

**1 Load spectra** — Files added via *Upload* are loaded and fitted automatically. *Load paths* is only for the textbox below it (one path per line, globs OK). Per-file checkboxes toggle each spectrum on/off. The *Explore* dropdown picks which file appears in the **Raw spectrum explorer** panel (it has no effect on the fit itself).

**2 Wavenumber calibration** (optional, off by default) — For the workflow where Hg(Ar) / Kr calibration lamps are on during acquisition so their sharp emission lines appear in every spectrum, letting each spectrum's x-axis be corrected on the fly. Tick *Use lamp calibration peaks* to enable. The textarea lists expected `label  wavenumber` rows (`#` for comments). *Search ±* sets the half-width of the window in which each line is hunted around its literature position. Keep it comfortably wider than the line FWHM (default ±8 cm⁻¹): if the window edges sit on the peak's wings the endpoint baseline subtraction biases the fitted centre (a too-narrow ±5 shifts positions by up to ~1.5 cm⁻¹), but not so wide it reaches a neighbouring line. A symmetric Gaussian is used deliberately — empirically it tracks the line's true mode to ~0.1 cm⁻¹ and is far more reproducible than asymmetric (skew) profiles, whose extra parameter inflates the position variance. Each row of the line list is `label  wavenumber  [wavelength_nm]` — the wavelength column is informational (shown in the diagnostic table); the wavenumber is what drives the correction. *Fit order* selects the polynomial mapping (measured − known) against wavenumber: *Auto (rule)* (1 line → offset, 2–3 → linear, 4+ → quadratic), ***Auto (AICc)*** (picks the order that minimises the small-sample-corrected Akaike criterion — the statistically-justified choice for a few noisy lines), or *Offset/Linear/Quadratic/Cubic* explicitly. **Don't choose the order by RMS** — RMS only ever falls with order. The diagnostic table reports, per order, **reduced χ²** (≈1 is good, <1 means overfitting), **AICc** and **LOOCV** (both minimised at the justified order, ◀ marks the one in use). A **diagnostic plot** to the right shows measured-vs-known (left) and fit residuals (right), each wrapped in a 2σ band propagated from each line's *measurement uncertainty* (not the residuals — so it survives exact fits and **balloons in extrapolation as the order rises**: the visual signature of overfitting). Watch for RMS improving while AICc/LOOCV worsen and the band blows up — that's fitting noise.

The calibration's own uncertainty is **propagated into the peak positions**: parameter tables carry a separate *Centre err (cal)* column (the systematic σ_cal at each peak, common-mode within a spectrum) alongside *Centre err (fit)* (the random fit error), and the summary plot folds σ_cal into the centre-axis error bars in quadrature (correct when comparing the same peak across spectra). **Extended-range (stitched) spectra** show a smooth wavenumber-dependent distortion that a single linear term can't remove — quadratic or cubic flattens it. *Allow single-peak offset* (on by default) lets the code fall back to a uniform shift when only one line is found — the typical single-segment B17 workflow. For each expected line a Gaussian is fitted within the search window and accepted only if its amplitude exceeds **10× the spectrum's noise σ**, so random noise spikes can't masquerade as missing calibration peaks. The correction `x − polyval(coeffs, x)` is applied to that file's whole x-axis. The default line list now spans the extended range: Hg(Ar) 479, Kr 813.2/839.3, and the Hg doublet at 1459.4/1522.4 cm⁻¹ (the latter two appear once spectra reach ~1500 cm⁻¹; values are for a 532 nm laser). Calibration peaks are excluded from Raman peak detection but stay visible in all plots as <span style="color:#1f77b4">**blue dashed**</span> vertical guides. The plot y-axes also auto-zoom to the sample peaks with the (much taller) lamp lines masked out, so they run off the top rather than squashing the real peaks.

The per-file status line above the plot reports the outcome:

- <span style="color:#080">**✓ file.txt: 5/5 lines (quadratic), shift +X.XXX cm⁻¹ at midpoint, RMS X.XXX (max Y.YYY) cm⁻¹**</span> — all expected lines found and they fall cleanly on the fitted curve (RMS < 0.3). **RMS and max are the "do the lamp lines lie on a smooth curve?" check**: small values confirm the correction is good; a stubbornly large RMS that *doesn't* drop when you raise the fit order suggests a genuine stitch step rather than smooth distortion.
- <span style="color:#c80">**! file.txt: N/M lines (...), ...**</span> — applied, but with a caveat: fewer than all expected lines, or a large RMS. Worth raising the fit order or checking the data. When fewer than all the expected lines are used, the status now spells out **which** line was dropped and **why** (e.g. *unused: Hg(Ar) 479 [SNR 6.1 < 10]*), and the diagnostic table grows a **status** column showing the same per line — the SNR for each line used, and the rejection reason (SNR below the gate, no data in the search window, or a failed fit) in red for each one dropped. So a calibration line that is present in the spectrum but not used is never silently missing.
- <span style="color:#c80">**! file.txt: 1/M lines (offset only), shift +X.XXX cm⁻¹**</span> — one line found; a uniform shift was applied (needs *Allow single-peak offset* ticked). An *N-point fit at exactly order N−1* (e.g. 3 lines + quadratic) passes exactly through every point, so RMS is trivially ~0 and the message shows only the midpoint shift.
- <span style="color:#c00">**✗ file.txt: 0/M lines — too few to correct**</span> — no correction applied; the spectrum's x-axis is left at its loaded values.

**3 Background subtraction** — *arPLS* (default) is usually the better choice. **Each spectrum gets its own baseline** — backgrounds are *not* shared between files. *Smoothness λ* controls baseline stiffness; raise it if the baseline cuts through peaks, lower it if it doesn't follow the background. *Edge trim* drops noisy edge points **before** the baseline is fitted — important for extended-range scans whose first few points ramp up instrumentally, which would otherwise drag the arPLS baseline far below the real signal and create a false pedestal under the low-wavenumber peaks.

**4 Peak fitting — two-pass strategy** — Peak detection runs **independently on every loaded file**, and the centres are then merged into a *union* peak list (centres within a few sample-spacings of each other are treated as the same peak). Every spectrum is then fitted to that same union, with each fit's initial amplitudes/centres taken from its **own** data — so a spectrum that contains only a subset of the dataset's peaks will simply have the absent peaks fitted to noise and blanked out in the table. This means the same N peak columns appear across all rows of the multi-file table, even when the files themselves are very different (e.g. a transect across a layered sample). The readout next to the slider shows the union count plus per-file counts so you can see how heterogeneous the dataset is.

Three per-spectrum gates decide which peaks survive into the results table:

- *Threshold %* — minimum peak height as a percentage of that spectrum's data range (calibration-lamp regions are masked from that range so a bright lamp line can't push the threshold above genuine smaller Raman peaks). Drawn as a horizontal grey dashed line on the single-spectrum main plot.
- *Width %* (range) — min and max FWHM as percentages of the current x-range. The **max** catches `curve_fit` runaway-broadening fits where a pseudo-Voigt swallowed noise; the **min** catches the symmetric failure where the FWHM collapsed onto a noise spike, producing a tall narrow false peak. Both ends are drawn as horizontal grey dashed lines on the summary plot.
- A 5σ noise floor sits on top of *Threshold %* so peakless / dark spectra detect zero peaks regardless of the slider.

*x range* restricts the fit window. *Waterfall offset* sets the y-spacing between spectra in the multi-spectrum view (cosmetic only). For Raman, *Pseudo-Voigt (shared η)* is usually the most stable choice. **Manual peaks:** the detector only finds local maxima, so it can't see a *shoulder* on the side of a bigger peak (e.g. the broad component under a sharp ν1, or a blended lattice mode). List extra centres (cm⁻¹, one per line) in the *Manual peaks* box to force-fit them — they're merged into the peak set, fitted at their given position (not snapped to a maximum), and still subject to the amplitude/width gates. Or tick *Auto-add peaks from residual* to do this automatically: peaks in the fit residual that clear 5σ and sit more than a cluster width from any existing peak are added and the spectra refitted (the distance test rejects the derivative-shaped residual of a merely mis-modelled peak). A single residual pass is used deliberately — it catches every genuinely-separated missing peak at once, whereas iterating tends to over-decompose an asymmetric band into extra components chasing its non-Pseudo-Voigt tail. Off by default since it changes the peak count for you. 

**Per-file peak gate** (optional, off by default) — by default every spectrum is fitted to the *whole* union peak list, so a peak present in only one or two files is still force-fitted into the rest, where it can steal intensity from a real neighbour and shift its centre/FWHM (this is the main reason a spectrum's fit can change when you add or remove other files, or crop the x-range). Tick *Per-file peak gate (skip absent peaks)* to fit each spectrum **only** to the union peaks that actually show a bump (≥5σ) in that spectrum — manually-added peaks are always kept. Absent peaks are left out of the fit entirely (not just blanked afterwards), so they can no longer perturb their neighbours, and the results become essentially independent of how many files are loaded and of the x-range. The readout appends *· per-file gate on*. Trade-off: for a peak that fades in *gradually* along a transect, it switches on at the file where it first clears 5σ, which can make a smooth trend look slightly stepped — so it's opt-in rather than the default.

The three plot panels are stacked top-to-bottom with the same width and x-axis, so peak x-positions line up vertically:

- **Raw spectrum explorer** — raw data + fitted baseline + modelled peaks for the file selected via *Explore*.
- **Waterfall / single-spectrum view** — all visible spectra (subtracted) plus a residual strip beneath.
- **Summary scatter** (Section 5) — peak position vs FWHM, same x-axis as the explorer.

The **single-spectrum** parameter table only lists peaks that survived the gates for that file. When more than one spectrum is visible the table widens to a *file × peak* grid with NaN cells where a peak was blanked. Peak labels (in the table, in the *Section 5* checkboxes, and in the *Section 6* end-member table) are **sequential among useful peaks** — i.e. if the union peak detection found five candidates but one was blanked everywhere, the surviving four are labelled Peak 1 through Peak 4 rather than 1, 2, 3, 5. The label always matches the on-plot guide line count.

Three kinds of vertical marker on the waterfall / single-spec / residual plots:
- **<span style="color:#1f77b4">Blue dashed</span>** lines at calibration-lamp positions (when Section 2 is enabled).
- **Grey dotted** lines at the **union** peak centres — the consensus position across all visible spectra.
- **Short coloured stubs** under each waterfall trace at *that spectrum's own* fitted peak centres. Compare these against the grey dotted lines to see whether a peak has drifted between spectra (continuous shifts won't line up, discrete mixing will).

**5 Summary plot** — Tick the peaks to include below the plot. With more than five peaks the toggles wrap into a multi-column grid. The y-axis autoscales to the currently-ticked peaks.

**6 Spectral decomposition** (optional, off by default) — Factor the dataset of mixed spectra into a small set of *end-member spectra* and per-spectrum *abundance weights*. Uses Non-negative Matrix Factorisation (NMF) on the baseline-subtracted spectra of all *currently-visible* files. Both end-members and weights are forced non-negative. Tick *Enable spectral decomposition*, then dial the *Components* slider up from 2; the reconstruction R² climbs steeply while you're adding components that capture real variance and plateaus once you have enough — the elbow is roughly the number of distinct phases. *Normalise each spectrum (area = 1)* (on by default) prevents absolute brightness from dominating, so the algorithm sees relative composition. The end-member spectra plot is sized like the explorer so peak x-positions line up, with vertical dotted lines at the union Raman peak centres for indexing; the per-spectrum weights plot is a stacked bar (one bar per spectrum, segments coloured by component, summing to 1).

The status line above the plots reports:

- <span style="color:#080">**N_spectra × N_wavenumbers, NMF n=K, R² = 0.XXX**</span> — green when R² > 0.95, amber for 0.85–0.95, red below.
- <span style="color:#888">**Need at least 2 visible spectra with overlapping x-axes**</span> — too few files visible, or their x-axes don't overlap.

A **parameter table** beneath the weights bar shows, for each end-member, the fitted centre / FWHM / amplitude of each useful union peak. Useful for diagnosing degenerate decompositions: if two components share the same peak positions and widths, they're not really independent phases — they're a linear split of the same underlying Raman signal (which can happen when the dataset's variation is dominated by *continuous* shifts in peak position rather than discrete mixing of fixed spectra; see notes below).

*Save end-members + weights* writes two TSV files: `endmembers_TIMESTAMP.txt` (columns: x and one column per component) and `weights_TIMESTAMP.txt` (rows: spectra, columns: components).

**7 Export** — *Save plot* writes the main figure. *Save data* writes a single-spectrum text file when only one is visible, otherwise a results-table TSV across all visible spectra.

*Quick start:* drop in files → defaults → if you used lamp calibration, tick *Use lamp calibration peaks* → tweak *Threshold %* until the readout's union peak count matches what you see → nudge the *Width %* range if obviously bogus narrow or broad fits sneak through.

In [ ]:
# Interactive Spectroscopy App  -  multi-file version
# ==============================================================================

# Internal state
S = dict(files=[], n_peaks=0, fig=None, fig2=None, _freeze=False)

# Uniform slider geometry so labels line up and tracks are the same length.
# Defined up-front so it's available to every section's widgets.
_SLIDER_LAYOUT = widgets.Layout(width='540px')
_SLIDER_STYLE  = {'description_width': '110px'}

# Style reset: make the FloatRangeSlider's editable readout look like the
# non-editable readout used by the single-value sliders (no border, no
# inset shadow, no input-field background). Without this the x range and
# Width % readouts get a thin grey "text box" appearance that doesn't match
# the Threshold % / Waterfall offset readouts.
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<style>
.widget-readout {
    border: none !important;
    box-shadow: none !important;
    background: transparent !important;
}
.widget-readout:focus { outline: none !important; }
</style>
"""))


# === 1. FILE LOADING ===
file_upload = widgets.FileUpload(
    accept='.txt,.csv,.dat,.asc', multiple=True,
    description='Upload', layout=widgets.Layout(width='220px'))
file_path_input = widgets.Textarea(
    value='', placeholder='Or paste one file path per line  (globs OK, e.g. /data/run_*.txt)',
    layout=widgets.Layout(width='560px', height='70px'))
load_btn = widgets.Button(
    description=' Load paths', button_style='primary', icon='paste',
    tooltip='Load files from the paths you typed/pasted into the text box. '
            'Files added with the Upload button are loaded automatically.',
    layout=widgets.Layout(width='130px'))
auto_cb = widgets.Checkbox(value=True, description='Auto-detect format', indent=False)
skip_input = widgets.IntText(
    value=0, description='Skip rows:',
    layout=widgets.Layout(width='160px'), style={'description_width': '70px'})
maxr_input = widgets.IntText(
    value=0, description='Max rows:',
    layout=widgets.Layout(width='160px'), style={'description_width': '70px'})
status_html = widgets.HTML(
    '<i style="color:#888">No data loaded. Upload files or paste paths above.</i>')

# Visibility checkbox container (rebuilt on every load)
files_box = widgets.VBox([], layout=widgets.Layout(
    border='1px solid #ddd', padding='4px',
    max_height='180px', overflow_y='auto', width='560px'))
ref_dropdown = widgets.Dropdown(
    options=[], description='Explore:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='560px'))
tick_all_btn   = widgets.Button(description='Tick all',   layout=widgets.Layout(width='100px'))
untick_all_btn = widgets.Button(description='Untick all', layout=widgets.Layout(width='100px'))


# === 2. WAVENUMBER CALIBRATION (optional) ===
# Designed for the workflow where Hg(Ar) / Kr calibration lamps are on
# during acquisition so their sharp emission lines appear in every spectrum.
# Off by default (most users won't be using it).
cal_use_cb = widgets.Checkbox(
    value=False, description='Use lamp calibration peaks',
    indent=False, layout=widgets.Layout(width='280px'))
cal_lines_input = widgets.Textarea(
    value=('# label   wavenumber(cm-1)   wavelength(nm)\n'
           'Hg(Ar)  479.0   546.07\n'
           'Kr      813.2   556.22\n'
           'Kr      839.2   557.03\n'
           'Hg(Ar)  1459.4  576.96\n'
           'Hg(Ar)  1522.4  579.07'),
    placeholder='label  wavenumber  [wavelength_nm]  (one per line, # comments)',
    layout=widgets.Layout(width='340px', height='118px'))
cal_window_slider = widgets.FloatSlider(
    value=8.0, min=1.0, max=20.0, step=0.5,
    description='Search \u00b1 (cm\u207b\u00b9):', readout_format='.1f',
    style={'description_width': '105px'},
    layout=widgets.Layout(width='320px'))
# Polynomial order of the wavenumber correction. 'Auto' picks a safe order
# from the number of lines found (1->offset, 2-3->linear, 4+->quadratic).
# Extended-range / stitched spectra need quadratic or cubic to remove the
# smooth wavenumber-dependent distortion the stitching introduces.
cal_order_dd = widgets.Dropdown(
    options=[('Auto (rule)', 'auto'), ('Auto (AICc)', 'aicc'),
             ('Offset (0)', 0), ('Linear (1)', 1),
             ('Quadratic (2)', 2), ('Cubic (3)', 3)],
    value='auto', description='Fit order:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='260px'))
# When only one expected line is found in a given spectrum, fall back to a
# zero-slope (uniform offset) correction. The typical Diamond/B17 workflow uses
# a single peak, so this is on by default.
cal_allow_offset_cb = widgets.Checkbox(
    value=True,
    description='Allow single-peak offset',
    indent=False,
    layout=widgets.Layout(width='320px'))
cal_status_html = widgets.HTML('')
# Diagnostic plot: measured-vs-known lamp positions + fit + residuals, with a
# measurement-error confidence band that balloons under overfitting.
cal_diag_out = widgets.Output(layout=widgets.Layout(max_width='680px'))
# Reference + model-selection table for the explored spectrum (lamp lines with
# wavelength, and per-order fit statistics so you can judge the order choice).
cal_table_html = widgets.HTML('', layout=widgets.Layout(max_width='760px'))
# Container that hides everything except the checkbox when calibration is off
cal_body = widgets.VBox(
    [widgets.HBox(
        [cal_lines_input,
         widgets.VBox([cal_window_slider, cal_order_dd, cal_allow_offset_cb],
                      layout=widgets.Layout(margin='0 0 0 18px',
                                            min_width='340px'))],
        layout=widgets.Layout(align_items='flex-start')),
     cal_diag_out,        # diagnostic plot below the controls
     cal_table_html,      # the two reference / stats tables side by side below
     cal_status_html],
    layout=widgets.Layout(margin='4px 0 0 0', display='none'))
# Common style for the vertical guides drawn at calibration-line positions.
_CAL_LINE = dict(color='#1f77b4', linestyle='--', linewidth=0.8,
                 alpha=0.45, zorder=0.4)

def _toggle_cal_visibility():
    cal_body.layout.display = '' if cal_use_cb.value else 'none'


# === 3. BASELINE ===
bl_algo = widgets.RadioButtons(
    options=['arPLS', 'Polynomial'], value='arPLS',
    description='',
    # 200 px matches profile_rb so the slider columns of all sections start at
    # the same x-position on the page.
    layout=widgets.Layout(width='200px'))

lam_slider = widgets.FloatLogSlider(
    value=1e4, base=10, min=1, max=10, step=0.1,
    description='Smoothness \u03bb:', readout_format='.1e',
    style=_SLIDER_STYLE, layout=_SLIDER_LAYOUT)
poly_deg = widgets.IntSlider(
    value=5, min=1, max=20, description='Poly degree:',
    style=_SLIDER_STYLE,
    layout=widgets.Layout(width='540px', display='none'))
trim_slider = widgets.IntSlider(
    value=5, min=0, max=100, step=1, description='Edge trim:',
    style=_SLIDER_STYLE, layout=_SLIDER_LAYOUT)


# === 3. PEAK FITTING ===
profile_rb = widgets.RadioButtons(
    options=['Gaussian', 'Lorentzian', 'Pseudo-Voigt',
             'Pseudo-Voigt (shared \u03b7)'],
    value='Pseudo-Voigt', description='',
    layout=widgets.Layout(width='200px'))
ht_slider = widgets.FloatLogSlider(
    value=10, base=10, min=-1, max=2, step=0.05,
    description='Threshold %:', readout_format='.1f',
    style=_SLIDER_STYLE, layout=_SLIDER_LAYOUT)
# Live readout of how many peaks the detector found, plus a hint if the
# slider was overridden by the noise floor or the MAX_PEAKS cap was hit.
peak_count_html = widgets.HTML(
    value='<span style="color:#888">-</span>',
    layout=widgets.Layout(margin='0 0 0 8px'))
range_slider = widgets.FloatRangeSlider(
    value=[0.0, 1.0], min=0.0, max=1.0, step=0.01,
    description='x range:',
    # .0f keeps the readout compact so the slider track lines up with the
    # other sliders. For typical Raman wavenumbers integer precision is fine;
    # the actual fit window is still set at the slider's full step precision.
    readout_format='.0f',
    style=_SLIDER_STYLE, layout=_SLIDER_LAYOUT)
range_reset_btn = widgets.Button(
    description='Reset range', icon='arrows-alt-h',
    layout=widgets.Layout(width='120px'))
offset_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=4.0, step=0.05,
    description='Waterfall offset:', readout_format='.2f',
    style=_SLIDER_STYLE, layout=_SLIDER_LAYOUT)
# Extra peak centres the user adds by hand, for shoulders / blended peaks that
# aren't local maxima (so find_peaks can't see them). Merged into the union.
manual_peaks_input = widgets.Textarea(
    value='',
    placeholder='extra centres, cm\u207b\u00b9 (one per line) \u2014 for '
                'shoulders the auto-detector misses',
    layout=widgets.Layout(width='300px', height='66px'))
# Iteratively add peaks found in the fit residual (catches shoulders the
# local-maximum detector misses). Off by default.
residual_auto_cb = widgets.Checkbox(
    value=False, description='Auto-add peaks from residual',
    indent=False, layout=widgets.Layout(width='300px'))
# Per-file peak gate (opt-in): when on, each spectrum is fitted ONLY to the
# union peaks that have local support (a 5-sigma bump) in THAT spectrum, so
# a peak present only in other files can't be force-fitted here and steal
# intensity from a real neighbour. Absent peaks are scattered back as
# zero-amplitude so the table columns still line up. Off by default; see
# CLAUDE.md for the (transect-sensitivity) trade-off.
support_gate_cb = widgets.Checkbox(
    value=False, description='Per-file peak gate (skip absent peaks)',
    indent=False, layout=widgets.Layout(width='300px'))
# Width sanity check (range): reject any peak whose fitted FWHM falls outside
# [min, max] expressed as percentages of the current x-range.
#   - Max catches "curve_fit widened a pseudo-Voigt to swallow noise" fits.
#   - Min catches the symmetric failure mode where curve_fit COLLAPSES the
#     FWHM toward its lower bound on a noise spike, producing tall narrow
#     spikes that still pass the height-threshold check.
# Default ~ 0.1% min / 20% max -- for a 1600 cm-1 x-range that is roughly
# 1.6 to 320 cm-1, comfortably wider than real Raman peaks (5-100 cm-1) at
# either end.
width_slider = widgets.FloatRangeSlider(
    value=[0.1, 20.0], min=0.0, max=30.0, step=0.05,
    description='Width %:', readout_format='.2f',
    style=_SLIDER_STYLE, layout=_SLIDER_LAYOUT)


# === 4. SUMMARY PLOT ===
# Two dropdowns let you swap what's on each axis. Default is centre vs FWHM;
# for transects use spectrum-index on one axis to track peak position or
# width along the sample.
_SUMMARY_AXIS_OPTIONS = [('Peak centre', 'centre'),
                        ('FWHM',        'fwhm'),
                        ('Spectrum #',  'index')]
summary_x_dd = widgets.Dropdown(
    options=_SUMMARY_AXIS_OPTIONS, value='centre',
    description='X axis:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='260px'))
summary_y_dd = widgets.Dropdown(
    options=_SUMMARY_AXIS_OPTIONS, value='fwhm',
    description='Y axis:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='260px'))

# Peak toggles live below the summary plot as a grid of checkboxes (one per
# union peak). Multi-column wrap kicks in above 5 peaks rather than
# scrolling, which keeps every peak visible without hiding any.
peak_select_box = widgets.GridBox(
    children=[],
    layout=widgets.Layout(
        grid_template_columns='repeat(5, 220px)',
        width='1100px',
        margin='6px 0 0 0'))
save_summary_btn = widgets.Button(
    description=' Save summary plot', icon='image', button_style='success',
    layout=widgets.Layout(width='200px'))
summary_save_html = widgets.HTML('')


# === 5. EXPORT ===
save_plot_btn = widgets.Button(
    description=' Save plot', icon='image', button_style='success',
    layout=widgets.Layout(width='130px'))
save_data_btn = widgets.Button(
    description=' Save data', icon='download', button_style='success',
    layout=widgets.Layout(width='130px'))
save_html = widgets.HTML('')


# === 6. OUTPUT AREAS ===
# All three plot Outputs are capped at the same width (matching the params
# table) so the explorer / waterfall / summary line up vertically on the page —
# you can trace a peak's x-position straight down from one panel to the next.
explorer_out     = widgets.Output(layout=widgets.Layout(max_width='1100px'))
plot_out         = widgets.Output(layout=widgets.Layout(max_width='1100px'))
params_out       = widgets.Output(layout=widgets.Layout(max_width='1100px',
                                                        overflow_x='auto'))
summary_plot_out = widgets.Output(layout=widgets.Layout(max_width='1100px'))


# ------------------------------------------------------------------
# Helpers for the peak-toggle grid (rebuilt whenever the union peak
# count changes).
# ------------------------------------------------------------------
def _display_index_for(popt_idx):
    """Map an original union peak index to its sequential display index.
    Useful peaks are re-numbered 0..M-1 in the order they appear; non-useful
    peaks (universally blanked) don't get a display slot."""
    useful = S.get('useful_mask', [])
    disp = -1
    for j, u in enumerate(useful):
        if u:
            disp += 1
            if j == popt_idx:
                return disp
    return popt_idx


def _selected_peaks():
    """Union peak indices of currently-ticked checkboxes (numbering matches
    Peak K labels; non-useful peaks are skipped so the result may have gaps)."""
    return [getattr(cb, '_peak_idx', i)
            for i, cb in enumerate(peak_select_box.children) if cb.value]


def _kept_centres(fd, profile):
    """Fitted centres for peaks that survived the per-spectrum kept gates.
    Used to draw vertical guide lines on the single-spectrum panels."""
    popt = fd.get('popt')
    if popt is None:
        return []
    n      = count_peaks(popt, profile)
    kept   = fd.get('peak_kept', [True] * n)
    stride = 4 if profile == 'Pseudo-Voigt' else 3
    return [float(popt[j * stride + 1]) for j in range(n)
            if j < len(kept) and kept[j]]


# Common style for the peak-centre guide lines drawn on every plot.
_GUIDE_LINE = dict(color='0.55', linestyle=':', linewidth=0.7,
                   alpha=0.55, zorder=0.5)

def _rebuild_peak_checkboxes(n_pk):
    """Recreate the peak-toggle checkboxes for *useful* union peaks only.

    A peak is useful if at least one spectrum kept it; universally-blanked
    union peaks (read from S['useful_mask']) are skipped here so they don't
    show up as dead checkboxes. The numbering survives gaps (e.g. if peak 7
    was dropped you'll see ..., 6, 8). Each checkbox is tagged with its
    union peak index via the _peak_idx attribute so downstream code can
    look up popt/centre/etc. without renumbering."""
    useful  = S.get('useful_mask', [True] * n_pk)
    indices = [j for j in range(n_pk) if j < len(useful) and useful[j]]
    existing = [getattr(cb, '_peak_idx', None) for cb in peak_select_box.children]
    if existing == indices:
        return
    old_state = {getattr(cb, '_peak_idx', i): cb.value
                 for i, cb in enumerate(peak_select_box.children)}
    cbs = []
    for disp, j in enumerate(indices):
        cb = widgets.Checkbox(
            value=old_state.get(j, True),
            description=f'Peak {disp+1}', indent=False,
            layout=widgets.Layout(width='200px'))
        cb._peak_idx = j           # original union index (for popt lookup)
        cb._peak_display = disp    # sequential index (for matching labels)
        cb.observe(lambda c: update_summary_plot(), names='value')
        cbs.append(cb)
    n_cols = max(1, min(5, len(cbs)))
    peak_select_box.layout.grid_template_columns = f'repeat({n_cols}, 220px)'
    peak_select_box.children = tuple(cbs)


# ==============================================================================
# UI helpers
# ==============================================================================

def _toggle_bl():
    a = bl_algo.value
    lam_slider.layout.display = '' if a == 'arPLS' else 'none'
    poly_deg.layout.display   = '' if a == 'Polynomial' else 'none'

_toggle_bl()
bl_algo.observe(lambda c: (_toggle_bl(), update_plot()), names='value')


def _update_range_slider():
    """Recompute range slider limits from the union of loaded x-axes.

    Uses ``hold_sync`` so the frontend never sees an intermediate state
    where the bounds and the value disagree. Without this batching,
    JupyterLab occasionally ends up displaying ``value = (xmin, xmin)``
    because the value-update sync message arrives before the
    max-update sync message and gets clamped by the still-narrow bounds."""
    files = S['files']
    if not files:
        return
    tr = trim_slider.value
    xmins, xmaxs = [], []
    for fd in files:
        x_full = fd['x']
        if tr > 0 and 2 * tr < len(x_full):
            xmins.append(float(x_full[tr]))
            xmaxs.append(float(x_full[-tr-1]))
        else:
            xmins.append(float(x_full[0]))
            xmaxs.append(float(x_full[-1]))
    xmin, xmax = min(xmins), max(xmaxs)
    if xmin > xmax:
        xmin, xmax = xmax, xmin
    if xmin == xmax:
        xmax = xmin + 1.0
    step = (xmax - xmin) / 500.0
    with range_slider.hold_sync():
        # Widen the bounds first so the about-to-be-set value can't be
        # clamped by either side.
        range_slider.max = max(xmax, range_slider.max)
        range_slider.min = min(xmin, range_slider.min)
        # Set the desired value while the bounds are still wide enough.
        range_slider.value = (xmin, xmax)
        # Now narrow the bounds to exactly the data range. Each narrow
        # step is safe because value[i] already equals the new bound.
        range_slider.min  = xmin
        range_slider.max  = xmax
        range_slider.step = step


def _rebuild_files_box():
    """Recreate the per-file visibility checkboxes from S['files']."""
    cbs = []
    for fd in S['files']:
        cb = widgets.Checkbox(
            value=True, description=fd['name'], indent=False,
            layout=widgets.Layout(width='540px'))
        cb.observe(lambda c: update_plot(), names='value')
        cbs.append(cb)
    files_box.children = tuple(cbs)


def on_range_reset(btn):
    _update_range_slider()
    update_plot()

range_reset_btn.on_click(on_range_reset)


def on_tick_all(btn):
    S['_freeze'] = True
    for cb in files_box.children:
        cb.value = True
    S['_freeze'] = False
    update_plot()

def on_untick_all(btn):
    S['_freeze'] = True
    for cb in files_box.children:
        cb.value = False
    S['_freeze'] = False
    update_plot()

tick_all_btn.on_click(on_tick_all)
untick_all_btn.on_click(on_untick_all)


# ==============================================================================
# LOAD
# ==============================================================================

def _gather_load_paths():
    """Return list of (filepath_to_load, display_name) from upload widget + path text."""
    paths = []
    upl = file_upload.value
    if upl:
        items = list(upl) if isinstance(upl, (list, tuple)) else [upl[k] for k in upl]
        for item in items:
            if not isinstance(item, dict):
                continue
            fname = item.get('name')
            raw = item.get('content')
            if fname is None or raw is None:
                continue
            # Stage the upload to the OS temp directory (cross-platform:
            # /tmp on Linux/macOS, %TEMP% on Windows).
            tmp = os.path.join(tempfile.gettempdir(), f'_spec_{fname}')
            with open(tmp, 'wb') as f:
                f.write(bytes(raw) if isinstance(raw, (bytes, memoryview))
                        else raw.encode('utf-8'))
            paths.append((tmp, fname))
    for line in file_path_input.value.splitlines():
        line = line.strip()
        if not line:
            continue
        line = os.path.expanduser(line)
        matches = sorted(glob.glob(line)) if any(c in line for c in '*?[') else [line]
        for m in matches:
            paths.append((m, os.path.basename(m)))
    return paths


def on_load(btn):
    try:
        targets = _gather_load_paths()
        if not targets:
            status_html.value = '<b style="color:#c00">No files supplied.</b>'
            return

        new_files, errors = [], []
        for fp, fn in targets:
            try:
                if auto_cb.value:
                    x, y, _, _ = auto_load_spectrum(fp)
                else:
                    sk = skip_input.value
                    mr = maxr_input.value if maxr_input.value > 0 else None
                    data = np.loadtxt(fp, skiprows=sk, max_rows=mr)
                    x, y = data[:, 0], data[:, 1]
                x_arr = np.asarray(x, float)
                new_files.append(dict(name=fn, x=x_arr.copy(),
                                      x_raw=x_arr,         # immutable original
                                      y=np.asarray(y, float)))
            except Exception as e:
                errors.append(f'{fn}: {e}')

        if not new_files:
            status_html.value = (f'<b style="color:#c00">All loads failed: '
                                 f'{"; ".join(errors[:3])}</b>')
            return

        # _freeze suppresses the chain of observer-fired update_plot calls
        # that would otherwise see a transient mismatch between S['files']
        # (already replaced) and stale files_box.children / ref_dropdown
        # options. Without this, loading a smaller set of files after a
        # larger one crashes with "list index out of range".
        S['_freeze'] = True
        try:
            S['files'] = new_files
            min_len = min(len(fd['x']) for fd in new_files)
            trim_slider.max = max(min_len // 4, 1)
            _rebuild_files_box()
            ref_dropdown.options = [(fd['name'], i)
                                    for i, fd in enumerate(new_files)]
            ref_dropdown.value = 0
            _update_range_slider()
        finally:
            S['_freeze'] = False

        ranges = [fd['y'].max() - fd['y'].min() for fd in new_files]
        msg = (f'<b style="color:#080">Loaded {len(new_files)} spectra</b> '
               f'(longest: {max(len(fd["x"]) for fd in new_files)} pts; '
               f'shortest: {min_len} pts)')
        if errors:
            msg += (f' &nbsp; <span style="color:#c00">{len(errors)} failed: '
                    f'{"; ".join(errors[:3])}</span>')
        status_html.value = msg
        update_plot()
    except Exception as e:
        status_html.value = f'<b style="color:#c00">Error: {e}</b>'

load_btn.on_click(on_load)
file_upload.observe(lambda c: on_load(None), names='value')


# ==============================================================================
# CORE: process all spectra, then redraw
# ==============================================================================

def _augment_union_from_residuals(files, union_xs, prof, exclude_centres,
                                  max_iter=1, k_sigma=5.0):
    """Grow the union peak set with peaks found in the fit residual.
    A missing shoulder shows up as a coherent positive residual blob (a local
    max once the main peak is subtracted). A blob is added only if it clears
    k_sigma*noise AND is more than a cluster width (5*dx) from every existing
    peak -- the distance test rejects the derivative-shaped residual a merely
    mis-modelled peak leaves straddling its own position.

    Default max_iter=1 (a single residual pass): the first residual already
    exposes every genuinely-separated missing peak at once, whereas further
    iterations tend to over-decompose an asymmetric band, adding extra
    components to chase the leftover non-Pseudo-Voigt tail of a peak just
    added. Stops early when an iteration finds nothing new."""
    centres = [float(c) for c in np.asarray(union_xs, float)]
    for _ in range(max_iter):
        added = []
        for fd in files:
            xs = fd['x_trim']; ys = fd['y_sub']
            if len(xs) < 5:
                continue
            dx = float(np.median(np.diff(xs)))
            merge_tol = 5.0 * dx
            idxs = snap_to_local_max(xs, ys, np.array(sorted(centres)), search_pts=10)
            p0, lo, hi, fn, jac = build_initial_guess(xs, ys, idxs, prof, dx)
            fm = None
            if exclude_centres:
                fm = np.ones(len(xs), dtype=bool)
                for cx in exclude_centres:
                    fm &= np.abs(xs - cx) > LAMP_MASK_HW
            popt, _, fc = fit_spectrum(xs, ys, p0, lo, hi, fn, jac, fit_mask=fm)
            if popt is None:
                continue
            rr = ys - fc
            if fm is not None:
                rr = np.where(fm, rr, 0.0)
            sig = estimate_noise_sigma(ys)
            ridx, _ = find_peaks(rr, height=k_sigma * sig,
                                 prominence=k_sigma * sig, width=2)
            for i in ridx:
                xc = float(xs[i])
                allc = centres + added
                if not allc or min(abs(xc - c) for c in allc) > merge_tol:
                    added.append(xc)
        if not added or len(centres) + len(added) > MAX_PEAKS:
            break
        centres += added
    return np.sort(np.array(centres))[:MAX_PEAKS]


def _scatter_popt(popt_sub, perr_sub, sup, n_pk, prof, union_xs, dx):
    """Expand a subset fit (only the union peaks listed in `sup`) back to a
    full-length parameter vector in the standard layout, with the absent
    peaks set to zero amplitude (centre pinned to the union centre, a
    nominal width) so every downstream gate / plot / table indexes it
    exactly as it would an ungated fit."""
    shared = (prof == 'Pseudo-Voigt (shared \u03b7)')
    stride = 4 if prof == 'Pseudo-Voigt' else 3
    size = n_pk * stride + (1 if shared else 0)
    popt = np.zeros(size)
    perr = np.full(size, np.nan)
    if shared:
        popt[-1] = popt_sub[-1]
        if perr_sub is not None:
            perr[-1] = perr_sub[-1]
    for m, j in enumerate(sup):
        popt[j*stride:j*stride+stride] = popt_sub[m*stride:m*stride+stride]
        if perr_sub is not None:
            perr[j*stride:j*stride+stride] = perr_sub[m*stride:m*stride+stride]
    for j in range(n_pk):
        if j not in sup:
            popt[j*stride+1] = union_xs[j]      # centre at the union position
            popt[j*stride+2] = 5.0 * dx         # nominal width (amp stays 0)
            if prof == 'Pseudo-Voigt':
                popt[j*stride+3] = 0.5
    return popt, perr


def _fit_supported(fd, union_xs, peak_idx, prof, dx, merge_tol,
                   exclude_centres, manual_mask):
    """Fit a spectrum to only the union peaks with local support in it.

    Support = a peak detected at the bare 5-sigma noise floor (detect_peaks
    with height_pct=0) lies within merge_tol of the union centre. Manual
    peaks are always kept. Returns (popt, perr, fit_curve, supported_list)
    in the same full-length layout an ungated fit produces; unsupported
    peaks are zero-amplitude and get blanked by the usual peak_kept gates,
    so any real signal wrongly dropped still surfaces in the residual."""
    n_pk = len(union_xs)
    sidx, _ = detect_peaks(fd['x_trim'], fd['y_sub'], 0.0,
                           exclude_centres=exclude_centres,
                           exclude_window=LAMP_MASK_HW)
    scen = fd['x_trim'][sidx]
    sup = []
    for j in range(n_pk):
        is_manual = (manual_mask is not None and j < len(manual_mask)
                     and manual_mask[j])
        has_signal = (len(scen) > 0
                      and float(np.min(np.abs(scen - union_xs[j]))) <= merge_tol)
        if is_manual or has_signal:
            sup.append(j)
    supported = [j in set(sup) for j in range(n_pk)]
    if not sup:
        return None, None, np.zeros_like(fd['x_trim']), supported
    red_idx = peak_idx[np.array(sup)]
    p0, lo, hi, func, jac = build_initial_guess(fd['x_trim'], fd['y_sub'],
                                                red_idx, prof, dx)
    fit_mask = None
    if exclude_centres:
        fit_mask = np.ones(len(fd['x_trim']), dtype=bool)
        for cx in exclude_centres:
            fit_mask &= np.abs(fd['x_trim'] - cx) > LAMP_MASK_HW
    popt_sub, perr_sub, _ = fit_spectrum(fd['x_trim'], fd['y_sub'],
                                         p0, lo, hi, func, jac,
                                         fit_mask=fit_mask)
    if popt_sub is None:
        return None, None, np.zeros_like(fd['x_trim']), supported
    popt, perr = _scatter_popt(popt_sub, perr_sub, sup, n_pk, prof,
                               union_xs, dx)
    fit_curve = eval_model(fd['x_trim'], popt, prof)
    return popt, perr, fit_curve, supported


def update_plot(*args):
    if S.get('_freeze') or not S['files']:
        return

    files = S['files']
    algo  = bl_algo.value
    lam   = lam_slider.value
    poly  = poly_deg.value
    tr    = trim_slider.value
    rng   = range_slider.value
    prof  = profile_rb.value
    ht    = ht_slider.value

    # 1.0 Per-file wavenumber calibration (optional).
    # Always derive fd['x'] from fd['x_raw'] so toggling off restores the
    # original axis cleanly.
    cal_on = cal_use_cb.value
    cal_lines = parse_cal_lines(cal_lines_input.value) if cal_on else []
    cal_window = float(cal_window_slider.value)
    exclude_centres = ([wn for _, wn, _ in cal_lines] if cal_on else None)
    if cal_on:
        status_parts = []
        for fd in files:
            x_raw = fd.get('x_raw')
            if x_raw is None:
                x_raw = fd['x'].copy()
                fd['x_raw'] = x_raw
            found = fit_calibration_lines(x_raw, fd['y'], cal_lines, cal_window)
            cal = compute_wavenumber_correction(
                found, allow_offset=cal_allow_offset_cb.value,
                order=cal_order_dd.value)
            n_expected = len(cal_lines)
            if cal is not None:
                coeffs = cal['coeffs']; rms = cal['rms']
                n_used = cal['n_used']; order_used = cal['order']
                resid = cal['residuals']
                fd['x'] = apply_wavenumber_correction(x_raw, coeffs)
                mode = {0: 'offset', 1: 'linear',
                        2: 'quadratic', 3: 'cubic'}[order_used]
                x_mid = float((x_raw.min() + x_raw.max()) / 2)
                shift_mid = float(np.polyval(coeffs, x_mid))
                max_resid = (float(np.max(np.abs(resid)))
                             if resid is not None and len(resid) else 0.0)
                # cal carries coeffs/order/rms/residuals/stats + the
                # covariance machinery (x0,s,invVtV,sigma) for sigma_cal(x).
                fd['cal_info'] = dict(cal, found=found, mode=mode,
                                      n_expected=n_expected,
                                      max_resid=max_resid)
                # Green only when every expected line was used AND they fall
                # cleanly on the fitted curve (small RMS). A large RMS means
                # the lamp lines don't lie on the chosen curve -- raise the
                # fit order, or suspect a genuine stitch step.
                # Green only for a genuine (order>=1) fit through all
                # expected lines that lands cleanly on the curve. Offset-only
                # (order 0) can't be verified from a single point, so it's
                # always flagged amber as a caveat.
                well_fit = rms < 0.3
                full = (n_used == n_expected) and well_fit and order_used >= 1
                colour = '#080' if full else '#c80'
                tick   = '\u2713' if full else '!'
                exact  = (n_used == order_used + 1)   # RMS trivially ~0
                if mode == 'offset':
                    lead = ('1/' if n_used == 1 else f'{n_used}/')
                    kind = ('offset only' if n_used == 1 else 'uniform offset')
                    detail = (f'{lead}{n_expected} lines ({kind}), '
                              f'shift {shift_mid:+.3f} cm\u207b\u00b9')
                elif exact:
                    detail = (f'{n_used}/{n_expected} lines ({mode}, '
                              f'shift {shift_mid:+.3f} cm\u207b\u00b9 '
                              f'at midpoint)')
                else:
                    detail = (f'{n_used}/{n_expected} lines ({mode}), '
                              f'shift {shift_mid:+.3f} cm\u207b\u00b9 '
                              f'at midpoint, RMS {rms:.3f} '
                              f'(max {max_resid:.3f}) cm\u207b\u00b9')
                unused = [d for d in found if not d['success']]
                if unused:
                    detail += (' \u2014 unused: ' + '; '.join(
                        f'{d["label"]} {d["known"]:.0f} [{cal_line_note(d)[0]}]'
                        for d in unused))
                status_parts.append(
                    f'<span style="color:{colour}">{tick} {fd["name"]}: '
                    f'{detail}</span>')
            else:
                n_found = sum(1 for d in found if d['success'])
                fd['x'] = x_raw.copy()
                fd['cal_info'] = None
                status_parts.append(
                    f'<span style="color:#c00">\u2717 {fd["name"]}: '
                    f'{n_found}/{n_expected} lines \u2014 too few to correct</span>')
        cal_status_html.value = '<br>'.join(status_parts)
        _draw_cal_diagnostic()
    else:
        for fd in files:
            if fd.get('x_raw') is not None:
                fd['x'] = fd['x_raw'].copy()
            fd['cal_info'] = None
        cal_status_html.value = ''
        cal_table_html.value = ''
        with cal_diag_out:
            clear_output(wait=True)

    # 1. Per-file: edge-trim, then baseline + subtract, then x-range.
    # Trimming BEFORE the baseline matters: instrumental edge artefacts (e.g.
    # the detector ramp-up in the first few points of an extended-range scan)
    # would otherwise drag the arPLS baseline far below the real signal across
    # the whole low-wavenumber region, creating a false pedestal and spurious
    # peaks. Removing those points first lets the baseline sit under the data.
    for fd in files:
        x_full, y_full = fd['x'], fd['y']
        if tr > 0 and 2 * tr < len(x_full):
            sl = slice(tr, -tr)
        else:
            sl = slice(None)
        x_t  = x_full[sl]
        yr_t = y_full[sl]
        if algo == 'arPLS':
            bl_t = arpls(yr_t, lam=lam)
        else:
            bl_t = poly_baseline(x_t, yr_t, degree=poly)
        ys_t = yr_t - bl_t

        mask = (x_t >= rng[0]) & (x_t <= rng[1])
        if mask.sum() < 5:
            mask = np.ones(len(x_t), dtype=bool)
        fd['x_trim']     = x_t[mask]
        fd['y_raw_trim'] = yr_t[mask]
        fd['bl_trim']    = bl_t[mask]
        fd['y_sub']      = ys_t[mask]

    # 2. PASS 1 — detect peaks independently on every file. We can no longer
    #    assume one "reference" spectrum represents the whole dataset (a
    #    transect across very different materials may have nothing in common).
    ref_idx = ref_dropdown.value if ref_dropdown.value is not None else 0
    if ref_idx is None or ref_idx >= len(files):
        ref_idx = 0
    per_file_items = []  # one list of (centre, height) tuples per file
    per_file_info  = []
    per_file_counts = []
    for fd in files:
        # exclude_window matches the fit-mask half-width (LAMP_MASK_HW): a
        # peak we can't fit (its data is masked) must not enter the union.
        pk_idx, info = detect_peaks(fd['x_trim'], fd['y_sub'], ht,
                                    exclude_centres=exclude_centres,
                                    exclude_window=LAMP_MASK_HW)
        per_file_items.append(list(zip(fd['x_trim'][pk_idx],
                                       fd['y_sub'][pk_idx])))
        per_file_info.append(info)
        per_file_counts.append(int(len(pk_idx)))

    # PASS 1.5 — merge across files (cluster centres that lie within a few dx
    # of each other; tied clusters get the median as the union centre).
    dx_all = float(np.median(np.concatenate(
        [np.diff(fd['x_trim']) for fd in files])))
    merge_tol = 5.0 * dx_all
    union_xs = union_peak_centres(per_file_items, merge_tol, MAX_PEAKS)
    # Add user-specified manual peak centres (shoulders / blended peaks the
    # local-maximum detector can't find); skip any already covered by an auto
    # centre, then re-cap at MAX_PEAKS.
    manual = parse_manual_peaks(manual_peaks_input.value)
    if manual:
        keep = [mx for mx in manual
                if len(union_xs) == 0 or np.min(np.abs(union_xs - mx)) > merge_tol]
        if keep:
            union_xs = np.sort(np.concatenate([np.asarray(union_xs, float),
                                               keep]))[:MAX_PEAKS]
    # Optionally grow the set with peaks found in the fit residual (shoulders).
    if residual_auto_cb.value and len(union_xs):
        union_xs = _augment_union_from_residuals(
            files, union_xs, prof, exclude_centres)
    # Flag the manual centres so they aren't snapped to a local maximum during
    # fitting (a shoulder has no local max -- snapping would collide it with
    # the neighbouring main peak).
    manual_set = set(manual)
    S['manual_mask'] = np.array([any(abs(float(u) - m) < 1e-6 for m in manual_set)
                                 for u in union_xs], dtype=bool)
    n_pk = len(union_xs)
    S['n_peaks'] = n_pk
    S['union_xs'] = union_xs

    # Live readout: union count, with per-file breakdown and warnings.
    any_capped = any(inf['capped'] for inf in per_file_info)
    if n_pk == 0:
        peak_count_html.value = (
            '<span style="color:#888">0 peaks across all files '
            '(raise sensitivity?)</span>')
    elif any_capped:
        peak_count_html.value = (
            f'<span style="color:#c80"><b>{n_pk} peaks</b> '
            f'(per-file cap reached — raise threshold)</span>')
    else:
        if len(files) > 1:
            counts_str = ', '.join(str(c) for c in per_file_counts)
            extra = f' (union of {len(files)} files; per-file: {counts_str})'
        else:
            extra = ''
        gate_note = (' \u00b7 per-file gate on' if support_gate_cb.value
                     else '')
        peak_count_html.value = (
            f'<span style="color:#080"><b>{n_pk} peaks</b> '
            f'detected{extra}{gate_note}</span>')

    # 3. PASS 2 — fit every spectrum with per-file initial conditions, using
    #    the union centres as the shared topology. snap_to_local_max grounds
    #    each fit's amplitudes/centres in the spectrum's own data; peaks
    #    absent in a given spectrum end up small and get blanked by
    #    peak_kept further down.
    union_centres_arr = union_xs if n_pk > 0 else None
    for fd in files:
        if n_pk == 0:
            fd['popt'] = None; fd['perr'] = None
            fd['fit_curve'] = np.zeros_like(fd['x_trim'])
            fd['residual']  = fd['y_sub'].copy()
            fd['peak_kept'] = []
            continue
        fd_peak_idx = snap_to_local_max(fd['x_trim'], fd['y_sub'],
                                        union_centres_arr, search_pts=10)
        # Manual (shoulder) peaks: keep their exact centre rather than snapping
        # to a local maximum that belongs to a neighbouring peak.
        _mm = S.get('manual_mask')
        if _mm is not None and len(_mm) == len(union_centres_arr):
            for _j in np.where(_mm)[0]:
                fd_peak_idx[_j] = int(np.argmin(
                    np.abs(fd['x_trim'] - union_centres_arr[_j])))
        dx_fd = float(np.median(np.diff(fd['x_trim'])))
        if support_gate_cb.value:
            # Per-file gate: fit only the union peaks with local evidence in
            # THIS spectrum (see _fit_supported). Peaks present only in other
            # files are excluded from the fit (not just blanked afterwards),
            # so they can't shift a real neighbour. Scattered back to full
            # length with zero amplitude so the columns still line up.
            popt, perr, fit_curve, _sup = _fit_supported(
                fd, union_centres_arr, fd_peak_idx, prof, dx_fd,
                merge_tol, exclude_centres, S.get('manual_mask'))
            fd['support_n'] = int(sum(_sup))
        else:
            p0, lo, hi, func, jac = build_initial_guess(
                fd['x_trim'], fd['y_sub'], fd_peak_idx, prof, dx_fd)
            # Mask calibration-lamp regions out of the FIT so their large
            # unmodelled residuals can't dominate the cost or pull broad
            # spurious peaks into the gaps between bright lamp lines. The
            # model is still evaluated on the full axis for plotting.
            fit_mask = None
            if exclude_centres:
                fit_mask = np.ones(len(fd['x_trim']), dtype=bool)
                for cx in exclude_centres:
                    fit_mask &= np.abs(fd['x_trim'] - cx) > LAMP_MASK_HW
            popt, perr, fit_curve = fit_spectrum(fd['x_trim'], fd['y_sub'],
                                                 p0, lo, hi, func, jac,
                                                 fit_mask=fit_mask)
            fd['support_n'] = n_pk
        fd['popt'] = popt
        fd['perr'] = perr
        fd['fit_curve'] = fit_curve
        fd['residual']  = fd['y_sub'] - fit_curve

        # Per-spectrum gates so weak / absent / implausibly broad peaks blank
        # out in the table:
        #   amplitude >= Threshold % of this spectrum's data range, AND
        #   FWHM      <= Max width % of the current x-range.
        kept = []
        if popt is not None:
            dr_local = masked_data_range(fd['x_trim'], fd['y_sub'],
                                         exclude_centres, 8.0)
            h_abs = ht / 100.0 * dr_local
            x_lo, x_hi = range_slider.value
            # Fall back to this file's own x extent if the slider value is
            # degenerate (e.g. before the first sync settles), so the width
            # gate doesn't silently disable itself.
            if x_hi - x_lo < 1e-9 and len(fd['x_trim']):
                x_lo = float(fd['x_trim'].min())
                x_hi = float(fd['x_trim'].max())
            if x_hi > x_lo:
                fwhm_min = (x_hi - x_lo) * width_slider.value[0] / 100.0
                fwhm_max = (x_hi - x_lo) * width_slider.value[1] / 100.0
            else:
                fwhm_min, fwhm_max = 0.0, float('inf')
            stride = 4 if prof == 'Pseudo-Voigt' else 3
            for j in range(n_pk):
                amp_ok  = popt[j * stride] >= h_abs
                fwhm    = popt[j * stride + 2]
                fwhm_ok = fwhm_min <= fwhm <= fwhm_max
                kept.append(amp_ok and fwhm_ok)

            # Two complementary degeneracy checks on the fit results.
            #
            # 1. Width-aware overlap: if two peaks ended up within roughly
            #    half a FWHM of each other, curve_fit gave a degenerate
            #    split (the same physical peak fitted twice). Merge the
            #    smaller-amplitude one into the survivor so the displayed
            #    fit and the table show ONE peak with the correct total
            #    amplitude.
            # 2. Unlocalised peak: a fitted centre whose uncertainty (from
            #    perr) exceeds the peak's own FWHM is essentially unknown.
            #    This catches cases of rank-deficient covariance that
            #    survived check 1 (e.g. one of the pair has a tiny amp).
            cens = [popt[j * stride + 1] for j in range(n_pk)]
            amps = [popt[j * stride]     for j in range(n_pk)]
            fwhms = [abs(popt[j * stride + 2]) for j in range(n_pk)]
            merged_into = set()  # surviving peaks that absorbed a merge
            for j in range(n_pk):
                if not kept[j]:
                    continue
                for k in range(n_pk):
                    if k == j or not kept[k]:
                        continue
                    thresh = max(min(fwhms[j], fwhms[k]) / 2.0, dx_fd)
                    if (abs(cens[j] - cens[k]) < thresh
                            and amps[k] >= amps[j]):
                        # j loses; merge into k
                        popt[k * stride] += popt[j * stride]
                        amps[k]          += amps[j]
                        popt[j * stride]  = 0.0
                        amps[j]           = 0.0
                        # After merging, the pre-merge perr values are no
                        # longer meaningful (they came from the degenerate
                        # covariance). NaN them out so the table and the
                        # summary error bars don't lie.
                        if perr is not None:
                            perr[j * stride : j * stride + stride] = np.nan
                            perr[k * stride : k * stride + stride] = np.nan
                        kept[j] = False
                        merged_into.add(k)
                        break
            # Error-based degeneracy check: skip peaks that just received a
            # merge, because their perr still reflects the pre-merge
            # degenerate covariance and would be misleadingly large.
            if perr is not None:
                for j in range(n_pk):
                    if not kept[j] or j in merged_into:
                        continue
                    cen_err_j = float(perr[j * stride + 1])
                    if cen_err_j > fwhms[j]:
                        kept[j] = False
        else:
            kept = [False] * n_pk
        fd['peak_kept'] = kept

        # Apply the peak_kept gates to the curve that will be plotted: zero
        # out the amplitudes of blanked peaks so the explorer / main / per-peak
        # overlays don't draw them, and so the residual surfaces those signals
        # as unfit features instead of hiding them.
        if popt is not None:
            p_plot = np.asarray(popt, float).copy()
            stride = 4 if prof == 'Pseudo-Voigt' else 3
            for j in range(n_pk):
                if j < len(kept) and not kept[j]:
                    p_plot[j * stride] = 0.0
            fd['popt_plot'] = p_plot
            fd['fit_curve'] = eval_model(fd['x_trim'], p_plot, prof)
            fd['residual']  = fd['y_sub'] - fd['fit_curve']
        else:
            fd['popt_plot'] = None

    # 4.5 Useful-peak mask: a union peak is considered useful only if at
    # least one spectrum kept it. Peaks that ended up blanked in every file
    # are universally-blanked noise artefacts of the union step; hide them
    # from the UI rather than letting them clutter the table as all-NaN
    # columns / dead checkboxes.
    useful_mask = [False] * n_pk
    for fd in files:
        kept = fd.get('peak_kept', [])
        for j in range(min(n_pk, len(kept))):
            if kept[j]:
                useful_mask[j] = True
    S['useful_mask'] = useful_mask
    n_useful  = sum(useful_mask)
    n_dropped = n_pk - n_useful

    # Refresh the peak-count readout now that we know how many union peaks
    # actually survived the fit; the previous text was an upper bound.
    if n_pk > 0 and not any_capped:
        if len(files) > 1:
            counts_str = ', '.join(str(c) for c in per_file_counts)
            extra = f' (union of {len(files)} files; per-file: {counts_str}'
            if n_dropped:
                extra += f'; {n_dropped} blanked in every file'
            extra += ')'
        else:
            extra = ''
        peak_count_html.value = (
            f'<span style="color:#080"><b>{n_useful} peaks</b> '
            f'detected{extra}</span>')

    # 5. Visibility from checkboxes (defensively clamped to current file count
    # in case observer order somehow lets us reach here with mismatched state)
    visible_idxs = [i for i, cb in enumerate(files_box.children)
                    if cb.value and i < len(files)]

    # 6. Rebuild the peak-toggle checkbox grid (preserves ticks across rebuilds)
    _rebuild_peak_checkboxes(n_pk)

    _draw_explorer()
    _draw_main_plot(visible_idxs, ref_idx, prof, n_pk)
    _draw_params(visible_idxs, prof, n_pk)
    update_summary_plot()
    if decomp_use_cb.value:
        _run_decomposition()


# ==============================================================================
# DRAW: main plot
# ==============================================================================

# Shared figure geometry \u2014 all three of explorer / main / summary use the
# same width so they align on the page, and the explorer + summary share the
# same height so the points-line-up-with-peaks trick works visually.
FIG_W       = 11.0   # width in inches (~1100 px @ 100 dpi inline default)
PANEL_H     = 5.14   # height of an "explorer-sized" panel
RESIDUAL_H  = 1.3    # height of the residual strip beneath the main plot


def _plain_axes(ax, axes='x'):
    """Force plain decimal tick labels (no exponent, no axis offset). Big
    wavenumber values like 1500 should display as '1500', not '1.5e3' or
    '+1500' with an offset annotation."""
    try:
        ax.ticklabel_format(style='plain', useOffset=False, axis=axes)
    except (AttributeError, TypeError):
        pass


def _lamp_centres():
    """Wavenumbers of the active calibration-lamp lines (empty when cal off)."""
    if not cal_use_cb.value:
        return []
    return [wn for _, wn, _ in parse_cal_lines(cal_lines_input.value)]


def _masked_ylim(x, y, centres, window=8.0, pad=0.08):
    """(lo, hi) for y with +-window cm-1 around each centre removed and a
    fractional pad added, so tall calibration-lamp lines don't squash the
    sample peaks. None when nothing to mask / too little data remains."""
    if not centres:
        return None
    xa = np.asarray(x, float); ya = np.asarray(y, float)
    m = np.ones(len(xa), dtype=bool)
    for cx in centres:
        m &= np.abs(xa - cx) > window
    if m.sum() < 2:
        return None
    lo = float(ya[m].min()); hi = float(ya[m].max())
    if not (hi > lo):
        return None
    d = (hi - lo) * pad
    return lo - d, hi + d


def _blank_lamps(x, y, centres, window=6.0):
    """Copy of y with samples within +-window of each lamp centre set to NaN,
    so the (huge, meaningless) lamp-line residual spikes aren't drawn as
    clipped vertical streaks. The blue dashed guides still mark the lines."""
    if not centres:
        return y
    ya = np.asarray(y, float).copy()
    xa = np.asarray(x, float)
    for cx in centres:
        ya[np.abs(xa - cx) <= window] = np.nan
    return ya


def _draw_explorer():
    """Top panel: raw + baseline + fitted model (peaks drawn on top of the
    baseline so they sit where they would in the raw data). Always shows a
    single spectrum, picked by the 'Explore:' dropdown. Aspect ratio matches
    the main plot's single-spectrum middle panel so peak x-positions can be
    read straight across to the waterfall and the summary scatter."""
    files = S['files']
    with explorer_out:
        clear_output(wait=True)
        if not files:
            return
        sel = ref_dropdown.value
        if sel is None or sel >= len(files):
            sel = 0
        fd = files[sel]
        if fd.get('x_trim') is None:
            return

        x  = fd['x_trim']; yr = fd['y_raw_trim']; bl = fd['bl_trim']
        prof = profile_rb.value
        popt = fd.get('popt_plot')   # use the gated popt so blanked peaks don't draw

        fig, ax = plt.subplots(figsize=(FIG_W, PANEL_H))
        ax.plot(x, yr, 'k-', lw=0.8, label='Raw')
        ax.plot(x, bl, 'b-', lw=1.5, label='Baseline')
        if popt is not None:
            x_fine  = np.linspace(x.min(), x.max(), 2000)
            bl_fine = np.interp(x_fine, x, bl)
            fit_fine = eval_model(x_fine, popt, prof)
            ax.plot(x_fine, fit_fine + bl_fine, 'r-', lw=1.5,
                    label='Model (baseline + peaks)')
            for j in range(count_peaks(popt, prof)):
                pk = eval_single_peak(x_fine, popt, j, prof)
                ax.fill_between(x_fine, bl_fine, bl_fine + pk,
                                color='r', alpha=0.08)
                ax.plot(x_fine, bl_fine + pk, 'r-', lw=0.6, alpha=0.4)
        lo, hi = range_slider.value
        if hi > lo:
            ax.set_xlim(lo, hi)
        _yl = _masked_ylim(x, yr, _lamp_centres())   # ignore tall lamp lines
        if _yl:
            ax.set_ylim(*_yl)
        ax.set_xlabel('x'); ax.set_ylabel('Intensity')
        ax.set_title(f'Raw spectrum explorer  \u2014  {fd["name"]}  '
                     f'\u2014  baseline ({bl_algo.value})')
        ax.legend(fontsize=8, loc='best')
        # Calibration-line markers (in a distinct colour) — when active
        if cal_use_cb.value:
            for _label, kx, _wl in parse_cal_lines(cal_lines_input.value):
                ax.axvline(kx, **_CAL_LINE)
        # Vertical guides at this spectrum's fitted peak centres
        for xc in _kept_centres(fd, prof):
            ax.axvline(xc, **_GUIDE_LINE)
        ax.tick_params(which='both', top=True, right=True, direction='in')
        _plain_axes(ax, 'both')
        plt.tight_layout()
        S['fig_explorer'] = fig
        plt.show()


def _draw_cal_diagnostic():
    """Calibration check for the *explored* spectrum: measured-vs-known lamp
    positions with the applied fit (left) and residuals (right), each wrapped
    in a 2-sigma band propagated from the lamp-line measurement errors (so it
    survives exact fits and balloons under overfitting). Also fills
    cal_table_html with a reference table (label, wavelength, known/measured
    wavenumber, residual) and the per-order model-selection statistics."""
    with cal_diag_out:
        clear_output(wait=True)
        if not cal_use_cb.value:
            cal_table_html.value = ''
            return
        files = S.get('files', [])
        if not files:
            cal_table_html.value = ''
            return
        sel = ref_dropdown.value
        if sel is None or sel >= len(files):
            sel = 0
        fd = files[sel]
        info = fd.get('cal_info')
        if info is None:
            cal_table_html.value = ''
            print('No usable calibration fit for this spectrum.')
            return
        kn = info['knowns']; me = info['measured']; ce = info['cen_err']
        diff = me - kn
        order = info['order']

        xr = fd.get('x_raw')
        lo = min(kn.min(), float(xr.min()) if xr is not None else kn.min())
        hi = max(kn.max(), float(xr.max()) if xr is not None else kn.max())
        xq = np.linspace(lo, hi, 300)
        fit_diff = np.polyval(info['coeffs'], xq)
        band = 2.0 * cal_position_sigma(info, xq)          # 2 sigma
        resid = info['residuals']
        yerr = np.where(np.isfinite(ce), ce, 0.0)

        fig, (axL, axR) = plt.subplots(1, 2, figsize=(6.6, 2.6))
        axL.plot(xq, xq, color='0.7', lw=0.6, ls=':')
        axL.fill_between(xq, xq + fit_diff - band, xq + fit_diff + band,
                         color='r', alpha=0.15, lw=0)
        axL.plot(xq, xq + fit_diff, 'r-', lw=1.0)
        axL.errorbar(kn, me, yerr=yerr, fmt='ko', ms=4, capsize=2, lw=0.8)
        axL.set_xlabel('Known (cm$^{-1}$)'); axL.set_ylabel('Measured')
        axL.set_title(f'{info["mode"]} fit', fontsize=8)
        axR.fill_between(xq, -band, band, color='r', alpha=0.15, lw=0)
        axR.axhline(0, color='r', lw=1.0)
        axR.errorbar(kn, resid, yerr=yerr, fmt='ko', ms=4, capsize=2, lw=0.8)
        axR.set_xlabel('Known (cm$^{-1}$)'); axR.set_ylabel('Residual')
        axR.set_title(f'RMS {info["rms"]:.3f}, \u00b12\u03c3 band', fontsize=8)
        for ax in (axL, axR):
            ax.tick_params(which='both', top=True, right=True, direction='in',
                           labelsize=7)
            _plain_axes(ax, 'both')
        fig.suptitle(f'Calibration check \u2014 {fd["name"]}', fontsize=8)
        plt.tight_layout(rect=(0, 0, 1, 0.95))
        S['fig_cal_diag'] = fig
        plt.show()

    # --- reference + model-selection tables (HTML) ---
    th = ('style="padding:1px 7px;border-bottom:1px solid #ccc;'
          'text-align:right"')
    td = 'style="padding:1px 7px;text-align:right"'
    rows = []
    for d in info['found']:
        wl = d.get('wavelength')
        meas = f"{d['measured']:.2f}" if d['success'] else '\u2014'
        r = (f"{d['measured'] - d['known']:+.2f}" if d['success'] else '')
        note, rej = cal_line_note(d)
        rcol = ' style="color:#c00;font-weight:bold"' if rej else ''
        rows.append(
            f"<tr{rcol}><td {td.replace('right','left')}>{d['label']}</td>"
            f"<td {td}>{('%.2f' % wl) if wl else ''}</td>"
            f"<td {td}>{d['known']:.1f}</td><td {td}>{meas}</td>"
            f"<td {td}>{r}</td>"
            f"<td {td.replace('right','left')}>{note}</td></tr>")
    lines_tbl = (
        '<table style="border-collapse:collapse;font-size:11px;margin:2px 0">'
        f'<tr><th {th.replace("right","left")}>line</th>'
        f'<th {th}>&lambda; nm</th><th {th}>known</th>'
        f'<th {th}>meas</th><th {th}>m&minus;k</th>'
        f'<th {th.replace("right","left")}>status</th></tr>'
        + ''.join(rows) + '</table>')

    stats = info['stats']
    names = {0: 'offset', 1: 'linear', 2: 'quadratic', 3: 'cubic'}
    srows = []
    for o, st in sorted(stats.items()):
        sel_mark = ' \u25c0' if o == order else ''
        bold = ' style="font-weight:bold"' if o == order else ''
        aicc = '\u221e' if not np.isfinite(st['aicc']) else f"{st['aicc']:.1f}"
        loo = '\u2014' if not np.isfinite(st['loocv']) else f"{st['loocv']:.3f}"
        srows.append(
            f"<tr{bold}><td {td.replace('right','left')}>{names[o]}{sel_mark}"
            f"</td><td {td}>{st['rms']:.3f}</td>"
            f"<td {td}>{st['chi2_red']:.2f}</td>"
            f"<td {td}>{aicc}</td><td {td}>{loo}</td></tr>")
    stats_tbl = (
        '<table style="border-collapse:collapse;font-size:11px;margin:2px 0">'
        f'<tr><th {th.replace("right","left")}>order</th>'
        f'<th {th}>RMS</th><th {th}>&chi;&sup2;&#8341;</th>'
        f'<th {th}>AICc</th><th {th}>LOOCV</th></tr>'
        + ''.join(srows) + '</table>'
        '<div style="font-size:10px;color:#777">&chi;&sup2;&#8341;&asymp;1 '
        'good; &lt;1 overfit. AICc/LOOCV minimum = justified order '
        '(&#9664; = chosen).</div>')
    cal_table_html.value = (
        '<div style="display:flex; gap:30px; align-items:flex-start; '
        'flex-wrap:wrap">'
        '<div>' + lines_tbl + '</div><div>' + stats_tbl + '</div></div>')


def _draw_main_plot(visible_idxs, ref_idx, prof, n_pk):
    """Main plot: waterfall (or single-spectrum data + fit) plus a residual
    strip underneath. Same width as the explorer above and the summary below
    so peak positions can be traced vertically across the three panels.
    The reference / raw-spectrum explorer is drawn separately by
    _draw_explorer."""
    files = S['files']
    with plot_out:
        clear_output(wait=True)
        if not visible_idxs:
            fig, ax = plt.subplots(figsize=(FIG_W, 3))
            ax.text(0.5, 0.5, 'No spectra visible \u2014 tick at least one file above.',
                    ha='center', va='center', transform=ax.transAxes, color='#888')
            ax.set_axis_off()
            S['fig'] = fig
            plt.show()
            return

        visible = [files[i] for i in visible_idxs]
        ref = files[ref_idx] if (ref_idx is not None and ref_idx < len(files)) else visible[0]

        # Two panels: main + residual strip. Height of the main panel matches
        # the explorer for a "1-spectrum" view; grows with the number of
        # visible spectra in the waterfall.
        if len(visible) == 1:
            main_h = PANEL_H
        else:
            main_h = max(PANEL_H, min(13, 0.4 * len(visible) + PANEL_H))
        fig_h = main_h + RESIDUAL_H

        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(FIG_W, fig_h),
            gridspec_kw={'height_ratios': [main_h, RESIDUAL_H]}, sharex=True)

        # --- Main panel ---
        if len(visible) == 1:
            fd = visible[0]
            x  = fd['x_trim']; ys = fd['y_sub']
            popt = fd.get('popt_plot')   # gated copy, blanked peaks zeroed
            residual = fd['residual']
            x_fine = np.linspace(x.min(), x.max(), 2000)

            ax1.plot(x, ys, 'k-', lw=0.8, label='Data')
            if popt is not None:
                fit_fine = eval_model(x_fine, popt, prof)
                ax1.plot(x_fine, fit_fine, 'r-', lw=1.5, label='Fit')
                for j in range(count_peaks(popt, prof)):
                    pk = eval_single_peak(x_fine, popt, j, prof)
                    ax1.fill_between(x_fine, 0, pk, color='r', alpha=0.08)
                    ax1.plot(x_fine, pk, 'r-', lw=0.6, alpha=0.4)
            # Height threshold guide line (single-spec only — on the waterfall
            # each spectrum has a different absolute threshold so a single
            # line would be misleading).
            dr_local = masked_data_range(x, ys, _lamp_centres(), 8.0)
            h_abs = ht_slider.value / 100.0 * dr_local
            ax1.axhline(h_abs, color='grey', lw=2.0, ls='--', alpha=0.6,
                        label=f'Height threshold ({ht_slider.value:.1f}%)')
            ax1.set_ylabel('Intensity (arb. units)')
            n_kept = len(_kept_centres(fd, prof))
            title_npeaks = f'{n_kept} peak{"s" if n_kept != 1 else ""}'
            ax1.set_title(f'Background-subtracted  \u2014  {title_npeaks} ({prof})')
            ax1.legend(fontsize=8, loc='best')

            ax2.plot(x, _blank_lamps(x, residual, _lamp_centres()),
                     'k-', lw=0.7)
            if cal_use_cb.value:
                for _label, kx, _wl in parse_cal_lines(cal_lines_input.value):
                    ax1.axvline(kx, **_CAL_LINE)
                    ax2.axvline(kx, **_CAL_LINE)
            for xc in _kept_centres(fd, prof):
                ax1.axvline(xc, **_GUIDE_LINE)
                ax2.axvline(xc, **_GUIDE_LINE)
            _yl1 = _masked_ylim(x, ys, _lamp_centres())
            if _yl1:
                ax1.set_ylim(*_yl1)
            _yl2 = _masked_ylim(x, residual, _lamp_centres())
            if _yl2:
                ax2.set_ylim(*_yl2)
        else:
            # Waterfall + overlaid residuals
            _lc = _lamp_centres()
            max_range = max(masked_data_range(fd['x_trim'], fd['y_sub'], _lc, 8.0)
                            for fd in visible)
            offset_step = offset_slider.value * max_range
            cmap = plt.get_cmap('tab10') if len(visible) <= 10 else plt.get_cmap('viridis')
            for i, fd in enumerate(visible):
                colour = (cmap(i / max(len(visible) - 1, 1))
                          if len(visible) > 10 else cmap(i % 10))
                offset = i * offset_step
                x = fd['x_trim']; ys = fd['y_sub']
                is_ref = (fd is ref)
                label = fd['name'] + (' (explored)' if is_ref else '')
                ax1.plot(x, ys + offset, '-', color=colour,
                         lw=1.0 + (0.6 if is_ref else 0.0), label=label)
                if fd.get('popt_plot') is not None:
                    x_fine = np.linspace(x.min(), x.max(), 2000)
                    fit_fine = eval_model(x_fine, fd['popt_plot'], prof)
                    ax1.plot(x_fine, fit_fine + offset, '-',
                             color='red' if is_ref else colour,
                             lw=0.8, alpha=0.7)
                    ax2.plot(x, _blank_lamps(x, fd['residual'], _lc),
                             '-', color=colour, lw=0.7, alpha=0.85)

            ax1.set_ylabel('Intensity (offset)')
            n_use = sum(S.get('useful_mask', [True] * n_pk))
            title_npk = (f'{n_use} peak{"s" if n_use != 1 else ""} per spectrum'
                         if n_use else 'no peaks detected')
            ax1.set_title(f'Waterfall  \u2014  {len(visible)} spectra  \u2014  '
                          f'{title_npk} ({prof})')
            # Calibration-line markers first (under the union peak guides)
            if cal_use_cb.value:
                for _label, kx, _wl in parse_cal_lines(cal_lines_input.value):
                    ax1.axvline(kx, **_CAL_LINE)
                    ax2.axvline(kx, **_CAL_LINE)
            # Vertical guides at the *useful* union peak centres (peaks that
            # were kept by at least one spectrum) so dead union entries don't
            # litter the plot.
            uxs   = S.get('union_xs', [])
            umask = S.get('useful_mask', [True] * len(uxs))
            for jj, xc in enumerate(uxs):
                if jj < len(umask) and umask[jj]:
                    ax1.axvline(xc, **_GUIDE_LINE)
                    ax2.axvline(xc, **_GUIDE_LINE)
            # Per-spectrum peak-centre tick marks: short coloured vertical
            # ticks under each spectrum's baseline at THIS spectrum's fitted
            # peak centres. Lets you see at a glance whether a peak has
            # drifted between spectra, vs the consensus union centre above.
            stride_w = 4 if prof == 'Pseudo-Voigt' else 3
            tick_h   = offset_step * 0.10 if offset_step > 0 else 1.0
            for i, fd in enumerate(visible):
                if fd.get('popt') is None:
                    continue
                colour = (cmap(i / max(len(visible) - 1, 1))
                          if len(visible) > 10 else cmap(i % 10))
                offset_i = i * offset_step
                kept = fd.get('peak_kept', [True] * n_pk)
                for j in range(n_pk):
                    if j < len(umask) and not umask[j]:
                        continue
                    if j < len(kept) and not kept[j]:
                        continue
                    cen = fd['popt'][j * stride_w + 1]
                    # Drop a stub under the baseline of this spectrum's row
                    ax1.plot([cen, cen],
                             [offset_i - tick_h, offset_i],
                             color=colour, lw=1.5, alpha=0.9,
                             solid_capstyle='butt')

            # Zoom the y-axis to the sample peaks (lamp lines run off the top).
            lows, highs = [], []
            for i, fd in enumerate(visible):
                yl = _masked_ylim(fd['x_trim'], fd['y_sub'], _lc, pad=0.0)
                if yl is None:
                    yl = (float(fd['y_sub'].min()), float(fd['y_sub'].max()))
                off = i * offset_step
                lows.append(yl[0] + off); highs.append(yl[1] + off)
            if highs and max(highs) > min(lows):
                pad = (max(highs) - min(lows)) * 0.04
                ax1.set_ylim(min(lows) - max(pad, tick_h * 1.3), max(highs) + pad)
            rlo, rhi = [], []
            for fd in visible:
                yl = _masked_ylim(fd['x_trim'], fd['residual'], _lc, pad=0.0)
                if yl is not None:
                    rlo.append(yl[0]); rhi.append(yl[1])
            if rhi and max(rhi) > min(rlo):
                rpad = (max(rhi) - min(rlo)) * 0.1
                ax2.set_ylim(min(rlo) - rpad, max(rhi) + rpad)

        ax1.tick_params(which='both', top=True, right=True, direction='in')
        ax2.axhline(0, color='r', lw=0.5, ls='--')
        ax2.set_xlabel('x'); ax2.set_ylabel('Residual')
        ax2.tick_params(which='both', top=True, right=True, direction='in')

        # Pin the x-axis to the same range the explorer and summary use, so
        # the vertical peak-centre guides line up vertically across all three
        # panels. ax2 follows via sharex=True.
        lo_x, hi_x = range_slider.value
        if hi_x > lo_x:
            ax1.set_xlim(lo_x, hi_x)
        _plain_axes(ax1, 'both')
        _plain_axes(ax2, 'both')

        plt.tight_layout()

        # For waterfall plots, attach the spectrum legend BENEATH the residual
        # panel, spanning the full figure width. fig.legend (rather than
        # ax1.legend) avoids overlapping the residual panel that sits below ax1.
        if len(visible) > 1 and len(visible) <= 30:
            handles, labels = ax1.get_legend_handles_labels()
            if handles:
                ncol = min(max(1, len(handles)), 5)
                fig.legend(handles, labels, fontsize=7,
                           loc='lower center', bbox_to_anchor=(0.5, -0.02),
                           ncol=ncol, frameon=False)
                # Reserve space at the bottom for the legend
                fig.subplots_adjust(bottom=0.06 + 0.03 * ((len(handles) + ncol - 1) // ncol))

        S['fig'] = fig
        plt.show()


# ==============================================================================
# DRAW: parameters table
# ==============================================================================

def _draw_params(visible_idxs, prof, n_pk):
    files = S['files']
    visible = [files[i] for i in visible_idxs]
    with params_out:
        clear_output(wait=True)
        if not visible:
            print('No spectra visible.'); return
        if n_pk == 0:
            print('No peaks detected on the reference spectrum \u2014 lower the height threshold.')
            return

        if len(visible) == 1:
            fd = visible[0]
            df = build_single_file_df(fd, prof)
            if df is None:
                print('Fit failed \u2014 try adjusting parameters.'); return
            if len(df) == 0:
                print('No peaks survived the threshold / width filters '
                      'for this spectrum.')
                return
            if prof == 'Pseudo-Voigt (shared \u03b7)' and fd.get('popt') is not None:
                eta = fd['popt'][-1]
                char = ('Lorentzian' if eta > 0.7
                        else ('Gaussian' if eta < 0.3 else 'Mixed'))
                display(widgets.HTML(
                    f'<div style="margin:4px 0">Shared \u03b7 = '
                    f'<b>{eta:.4f}</b> &nbsp; ({char})</div>'))
        else:
            df = build_multi_file_df(visible, prof, n_pk,
                                     useful_mask=S.get('useful_mask'))

        itables_show(df.round(4), paging=False, scrollX=True,
                     classes='display compact', maxBytes=0,
                     style='width:100%; max-width:1080px')


# ==============================================================================
# DRAW: summary plot (centre vs FWHM with error bars)
# ==============================================================================

_SUMMARY_AXIS_LABEL = {'centre': 'Peak centre',
                       'fwhm':   'FWHM',
                       'index':  'Spectrum index'}

def _axis_value(axis, params, spec_index):
    """Return (value, error) for a peak's parameter dict on the chosen axis.
    'index' returns the 1-based spectrum index with zero error."""
    if axis == 'centre':
        return params['cen'], params.get('cen_err', 0.0)
    if axis == 'fwhm':
        return params['fwhm'], params.get('fwhm_err', 0.0)
    if axis == 'index':
        return float(spec_index), 0.0
    return None, None


def update_summary_plot(*args):
    files = S['files']
    n_pk = S['n_peaks']
    selected = _selected_peaks()
    visible_idxs = [i for i, cb in enumerate(files_box.children)
                    if cb.value and i < len(files)]
    visible = [files[i] for i in visible_idxs]
    x_axis = summary_x_dd.value
    y_axis = summary_y_dd.value

    with summary_plot_out:
        clear_output(wait=True)
        if not visible:
            print('No spectra visible.'); return
        if n_pk == 0 or not selected:
            print('Tick at least one peak below to plot '
                  f'{_SUMMARY_AXIS_LABEL[x_axis]} vs '
                  f'{_SUMMARY_AXIS_LABEL[y_axis]}.')
            return

        prof = profile_rb.value
        fig, ax = plt.subplots(figsize=(FIG_W, PANEL_H))
        cmap = plt.get_cmap('tab10')
        markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', 'h', '*']
        any_pts = False
        # Track x and y extent (including error bars) of *only the plotted
        # peaks* so both axes zoom in tighter when peaks are unticked.
        x_lo_acc, x_hi_acc = float('inf'), float('-inf')
        y_lo_acc, y_hi_acc = float('inf'), float('-inf')
        for k, j in enumerate(selected):
            xs, xerrs, ys, yerrs = [], [], [], []
            for spec_i, fd in enumerate(visible):
                params = peak_params_with_errors(
                    fd.get('popt'), fd.get('perr'), prof, n_pk)
                kept = fd.get('peak_kept', [True] * n_pk)
                if j < len(params) and params[j] is not None and (
                        j >= len(kept) or kept[j]):
                    xv, xe = _axis_value(x_axis, params[j], spec_i + 1)
                    yv, ye = _axis_value(y_axis, params[j], spec_i + 1)
                    # On a 'centre' axis, fold in this spectrum's systematic
                    # calibration uncertainty in quadrature: across spectra
                    # the per-file cal errors are independent, so they add to
                    # the random fit error when comparing the same peak.
                    cs = (float(cal_position_sigma(fd.get('cal_info'),
                                                   params[j]['cen']))
                          if fd.get('cal_info') else 0.0)
                    if x_axis == 'centre':
                        xe = float(np.hypot(xe, cs))
                    if y_axis == 'centre':
                        ye = float(np.hypot(ye, cs))
                    xs.append(xv); xerrs.append(xe)
                    ys.append(yv); yerrs.append(ye)
            if xs:
                any_pts = True
                # Use the same sequential numbering as the checkboxes/table
                # rather than the raw popt index, so legend / labels / table
                # all agree.
                disp = _display_index_for(j)
                ax.errorbar(xs, ys, xerr=xerrs, yerr=yerrs,
                            fmt=markers[k % len(markers)],
                            color=cmap(k % 10), capsize=2,
                            markersize=6, lw=0.8,
                            label=f'Peak {disp + 1}')
                for x_pt, xe, y_pt, ye in zip(xs, xerrs, ys, yerrs):
                    xe = xe if (xe is not None and np.isfinite(xe)) else 0.0
                    ye = ye if (ye is not None and np.isfinite(ye)) else 0.0
                    x_lo_acc = min(x_lo_acc, x_pt - xe)
                    x_hi_acc = max(x_hi_acc, x_pt + xe)
                    y_lo_acc = min(y_lo_acc, y_pt - ye)
                    y_hi_acc = max(y_hi_acc, y_pt + ye)

        # X-axis: auto-scale to the actually-plotted points so a single-peak
        # selection isn't shown as a thin vertical line on an empty wide axis.
        # Padding is 10% of the span (no fixed-cm floor) so subtle variations
        # in peak centre across spectra -- often just a fraction of a cm-1 --
        # actually show up as a visible spread rather than being swamped by
        # whitespace.
        #
        # When the axis is a wavenumber (centre), clamp it to the range
        # slider's window so a single peak with a runaway error bar can't
        # stretch the scale far beyond the actual data range.
        def _clamp_wn(lo, hi):
            if x_axis != 'centre':
                return lo, hi
            lo_rs, hi_rs = range_slider.value
            if hi_rs > lo_rs:
                return max(lo, lo_rs), min(hi, hi_rs)
            return lo, hi
        if any_pts and x_hi_acc > x_lo_acc:
            span_x = x_hi_acc - x_lo_acc
            pad_x  = max(0.10 * span_x, 0.05)
            lo_new, hi_new = _clamp_wn(x_lo_acc - pad_x, x_hi_acc + pad_x)
            ax.set_xlim(lo_new, hi_new)
        elif any_pts:
            lo_new, hi_new = _clamp_wn(x_lo_acc - 0.5, x_lo_acc + 0.5)
            ax.set_xlim(lo_new, hi_new)
        else:
            lo_rs, hi_rs = range_slider.value
            if hi_rs > lo_rs:
                ax.set_xlim(lo_rs, hi_rs)

        # Width sanity-check guide lines on whichever axis is FWHM
        # (skipped if neither axis is FWHM).
        lo, hi = range_slider.value
        if hi > lo:
            wmin, wmax = width_slider.value
            fwhm_min = (hi - lo) * wmin / 100.0
            fwhm_max = (hi - lo) * wmax / 100.0
            line_label = f'Width range ({wmin:.2f}-{wmax:.1f}%)'
            if y_axis == 'fwhm':
                ax.axhline(fwhm_min, color='grey', lw=2.0, ls='--',
                           alpha=0.6, label=line_label)
                ax.axhline(fwhm_max, color='grey', lw=2.0, ls='--', alpha=0.6)
            elif x_axis == 'fwhm':
                ax.axvline(fwhm_min, color='grey', lw=2.0, ls='--',
                           alpha=0.6, label=line_label)
                ax.axvline(fwhm_max, color='grey', lw=2.0, ls='--', alpha=0.6)

        # Explicit y autoscale: pad by 5% of the span (or 0.5 absolute,
        # whichever is larger), clamped at 0 since FWHM is non-negative.
        # When the unticked peaks contributed extreme FWHMs, this gives a
        # genuine "zoom in" on the remaining selection.
        def _clamp_wn_y(lo, hi):
            if y_axis != 'centre':
                return lo, hi
            lo_rs, hi_rs = range_slider.value
            if hi_rs > lo_rs:
                return max(lo, lo_rs), min(hi, hi_rs)
            return lo, hi
        if any_pts and y_hi_acc > y_lo_acc:
            span = y_hi_acc - y_lo_acc
            pad  = max(0.05 * span, 0.5)
            lo_new, hi_new = _clamp_wn_y(y_lo_acc - pad, y_hi_acc + pad)
            if y_axis != 'centre':
                lo_new = max(0.0, lo_new)
            ax.set_ylim(lo_new, hi_new)
        elif any_pts:
            c = max(y_lo_acc, 0.0)
            lo_new, hi_new = _clamp_wn_y(c - 1.0, c + 1.0)
            if y_axis != 'centre':
                lo_new = max(0.0, lo_new)
            ax.set_ylim(lo_new, hi_new)

        ax.set_xlabel(_SUMMARY_AXIS_LABEL[x_axis])
        ax.set_ylabel(_SUMMARY_AXIS_LABEL[y_axis])
        ax.set_title(f'{_SUMMARY_AXIS_LABEL[x_axis]} vs '
                     f'{_SUMMARY_AXIS_LABEL[y_axis]}  ({len(visible)} spectra)')
        if any_pts:
            ax.legend(fontsize=8, loc='best')
        else:
            ax.text(0.5, 0.5, 'No peaks pass the threshold for the selected files.',
                    ha='center', va='center', transform=ax.transAxes, color='#888')
        ax.tick_params(which='both', top=True, right=True, direction='in')
        from matplotlib.ticker import MaxNLocator
        if x_axis == 'index':
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        else:
            _plain_axes(ax, 'x')
        if y_axis == 'index':
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        else:
            _plain_axes(ax, 'y')
        plt.tight_layout()
        S['fig2'] = fig
        plt.show()


# ==============================================================================
# Wire everything that should retrigger a full update
# ==============================================================================

# Anything that affects the fit triggers a full update. The Explore dropdown
# only changes which spectrum the explorer panel shows — it doesn't influence
# the fit any more — so it just redraws the explorer (cheap).
for _w in [lam_slider, poly_deg, trim_slider, profile_rb, ht_slider,
           range_slider, width_slider]:
    _w.observe(lambda c: update_plot(), names='value')
manual_peaks_input.observe(lambda c: update_plot(), names='value')
residual_auto_cb.observe(lambda c: update_plot(), names='value')
support_gate_cb.observe(lambda c: update_plot(), names='value')
ref_dropdown.observe(lambda c: (_draw_explorer(), _draw_cal_diagnostic()), names='value')

# Wavenumber calibration controls: toggling reveals/hides the controls and
# re-runs the pipeline; editing lines or window only matters when cal is on.
cal_use_cb.observe(lambda c: (_toggle_cal_visibility(), update_plot()),
                   names='value')
def _on_cal_param_change(*_):
    if cal_use_cb.value:
        update_plot()
cal_lines_input.observe(_on_cal_param_change, names='value')
cal_window_slider.observe(_on_cal_param_change, names='value')
cal_allow_offset_cb.observe(_on_cal_param_change, names='value')
cal_order_dd.observe(_on_cal_param_change, names='value')
summary_x_dd.observe(lambda c: update_summary_plot(), names='value')
summary_y_dd.observe(lambda c: update_summary_plot(), names='value')

# Offset slider is purely cosmetic - cheaper to just redraw, but a full update is fine here
offset_slider.observe(lambda c: update_plot(), names='value')

trim_slider.observe(lambda c: _update_range_slider(), names='value')


# ==============================================================================
# Spectral decomposition (Section 6)
# ==============================================================================

# Master toggle - hides the heavy controls/plots when not in use
decomp_use_cb = widgets.Checkbox(
    value=False, description='Enable spectral decomposition',
    indent=False, layout=widgets.Layout(width='320px'))

# Brief inline explainer rendered above the controls (always visible).
decomp_intro_html = widgets.HTML(
    '<div style="max-width:1080px; margin:4px 0 6px; line-height:1.4; '
    'color:#333">'
    '<b>Spectral decomposition</b> separates a dataset of mixed spectra '
    'into a small number of <i>end-member spectra</i> (one per "phase") '
    'and a set of <i>abundance weights</i> per spectrum. Useful when many '
    'spectra share the same few underlying components in different '
    'proportions (transects across a layered sample, kinetics, etc.).'
    '<br><br>'
    'Uses <b>Non-negative Matrix Factorisation (NMF)</b> on the currently-'
    'visible spectra. Both end-members and weights are forced &ge; 0. '
    'Raise the components slider until R&#178; stops improving meaningfully '
    '&mdash; the plateau is roughly the number of distinct phases. Vertical '
    'dotted lines on the end-member plot mark the union Raman peak centres '
    'so you can see which components contain which peaks.</div>')

decomp_n_slider = widgets.IntSlider(
    value=3, min=2, max=8, step=1, description='Components:',
    style=_SLIDER_STYLE, layout=_SLIDER_LAYOUT)
decomp_normalize_cb = widgets.Checkbox(
    value=True,
    description='Normalise each spectrum (area = 1) before decomposing',
    indent=False, layout=widgets.Layout(width='540px'))
decomp_status_html = widgets.HTML('')
decomp_save_btn = widgets.Button(
    description=' Save end-members + weights', icon='download',
    button_style='success', layout=widgets.Layout(width='280px'))
decomp_save_html = widgets.HTML('')

# Output panels - same width as the explorer / main / summary plots so peaks
# line up vertically across the whole notebook page.
decomp_spectra_out = widgets.Output(layout=widgets.Layout(max_width='1100px'))
decomp_weights_out = widgets.Output(layout=widgets.Layout(max_width='1100px'))
decomp_params_out  = widgets.Output(layout=widgets.Layout(max_width='1100px',
                                                          overflow_x='auto'))

decomp_body = widgets.VBox(
    [decomp_n_slider,
     decomp_normalize_cb,
     decomp_status_html,
     decomp_spectra_out,
     decomp_weights_out,
     decomp_params_out,
     widgets.HBox([decomp_save_btn, decomp_save_html])],
    layout=widgets.Layout(display='none', margin='4px 0 0 0'))

def _toggle_decomp_visibility():
    decomp_body.layout.display = '' if decomp_use_cb.value else 'none'


# Colour map and style constants shared between the two output panels.
_DECOMP_CMAP = plt.get_cmap('tab10')

def _draw_decomp_spectra(x_common, H, r2, n_components, peak_centres):
    """Stacked-component plot of end-member spectra (one offset trace each)."""
    with decomp_spectra_out:
        clear_output(wait=True)
        if H is None:
            return
        fig, ax = plt.subplots(figsize=(FIG_W, PANEL_H))
        offset = float(H.max()) * 1.1 if H.max() > 0 else 0.1
        for k in range(n_components):
            colour = _DECOMP_CMAP(k % 10)
            ax.plot(x_common, H[k] + k * offset, '-', color=colour,
                    lw=1.0, label=f'Component {k+1}')
        # Union Raman peak centres
        for xc in peak_centres:
            ax.axvline(xc, **_GUIDE_LINE)
        lo, hi = range_slider.value
        if hi > lo:
            ax.set_xlim(lo, hi)
        ax.set_xlabel('x')
        ax.set_ylabel('End-member spectra (offset)')
        ax.set_title(f'End-member spectra  \u2014  {n_components} components, '
                     f'R\u00b2 = {r2:.4f}')
        ax.legend(fontsize=8, loc='upper right')
        ax.tick_params(which='both', top=True, right=True, direction='in')
        _plain_axes(ax, 'both')
        plt.tight_layout()
        S['fig_decomp_spectra'] = fig
        plt.show()


def _draw_decomp_weights(W, names, n_components):
    """Stacked-bar plot of per-spectrum component weights (each bar sums to 1)."""
    with decomp_weights_out:
        clear_output(wait=True)
        if W is None:
            return
        n_spectra = W.shape[0]
        fig, ax = plt.subplots(figsize=(FIG_W, PANEL_H))
        row_sums = W.sum(axis=1, keepdims=True)
        W_disp = W / np.maximum(row_sums, 1e-10)
        bottom = np.zeros(n_spectra)
        x = np.arange(n_spectra)
        for k in range(n_components):
            colour = _DECOMP_CMAP(k % 10)
            ax.bar(x, W_disp[:, k], bottom=bottom, color=colour,
                   label=f'Component {k+1}', width=0.85)
            bottom += W_disp[:, k]
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Fractional weight')
        ax.set_title(f'Component abundances per spectrum  '
                     f'({n_spectra} spectra, {n_components} components)')
        ax.legend(fontsize=8, loc='upper right',
                  bbox_to_anchor=(1.0, 1.0), borderaxespad=0., frameon=False)
        ax.set_ylim(0, 1.05)
        ax.tick_params(which='both', top=True, direction='in')
        _plain_axes(ax, 'y')
        plt.tight_layout()
        S['fig_decomp_weights'] = fig
        plt.show()


def _draw_decomp_params(x_common, H, n_components):
    """Fit each end-member to the union peak set and render a parameter
    table (rows = components, columns = peak * metric). Lets you see at a
    glance which peaks live in which component, with what centre and FWHM."""
    with decomp_params_out:
        clear_output(wait=True)
        uxs = S.get('union_xs')
        union_xs = list(uxs) if uxs is not None else []
        useful   = S.get('useful_mask', [True] * len(union_xs))
        kept_idx = [j for j in range(len(union_xs))
                    if j < len(useful) and useful[j]]
        kept_xs  = [union_xs[j] for j in kept_idx]
        if not kept_xs:
            print('No Raman peaks detected yet \u2014 nothing to tabulate.')
            return
        prof = profile_rb.value
        fits = fit_endmembers_to_peaks(x_common, H, kept_xs, prof)
        df = build_endmember_df(fits, prof, kept_idx)
        itables_show(df.round(4), paging=False, scrollX=True,
                     classes='display compact', maxBytes=0,
                     style='width:100%; max-width:1080px')


def _run_decomposition():
    """Run NMF on the currently-visible spectra and refresh both panels.
    Called whenever the toggle is on and either upstream data or the
    decomposition controls change."""
    if not decomp_use_cb.value:
        return
    files = S.get('files', [])
    visible_idxs = [i for i, cb in enumerate(files_box.children)
                    if cb.value and i < len(files)]
    visible = [files[i] for i in visible_idxs]
    x_common, Y, names = build_decomp_matrix(visible)
    if x_common is None:
        decomp_status_html.value = (
            '<span style="color:#888">Need at least 2 visible spectra with '
            'overlapping x-axes to decompose.</span>')
        with decomp_spectra_out: clear_output(wait=True)
        with decomp_weights_out: clear_output(wait=True)
        S['decomp'] = None
        return
    n_req = decomp_n_slider.value
    n_max = min(Y.shape) - 1
    n_components = max(2, min(n_req, n_max))
    if n_components < 2:
        decomp_status_html.value = (
            '<span style="color:#c00">Need more spectra/samples for the '
            'requested number of components.</span>')
        return
    try:
        H, W, r2 = decompose_nmf(Y, n_components,
                                 normalize=decomp_normalize_cb.value)
    except Exception as e:
        decomp_status_html.value = (
            f'<span style="color:#c00">NMF failed: {e}</span>')
        return
    S['decomp'] = dict(x_common=x_common, H=H, W=W, names=names, r2=r2,
                       n_components=n_components,
                       normalize=decomp_normalize_cb.value)
    capped = (n_req > n_components)
    colour = ('#080' if r2 > 0.95 else '#c80' if r2 > 0.85 else '#c00')
    note = (f' (capped from {n_req} to fit {Y.shape[0]} spectra)'
            if capped else '')
    decomp_status_html.value = (
        f'<span style="color:{colour}">{Y.shape[0]} spectra \u00d7 '
        f'{Y.shape[1]} wavenumbers, NMF n={n_components}{note}, '
        f'reconstruction R\u00b2 = {r2:.4f}</span>')
    # Use the useful union peak centres for the guide lines.
    # NB: union_xs is a numpy array, so test via `is not None` instead of
    # truthiness (ambiguous for arrays with >1 element).
    uxs = S.get('union_xs')
    union_xs = list(uxs) if uxs is not None else []
    useful   = S.get('useful_mask', [True] * len(union_xs))
    centres  = [xc for j, xc in enumerate(union_xs)
                if j < len(useful) and useful[j]]
    _draw_decomp_spectra(x_common, H, r2, n_components, centres)
    _draw_decomp_weights(W, names, n_components)
    _draw_decomp_params(x_common, H, n_components)


# Observers: run on toggle on, and on parameter changes while on
decomp_use_cb.observe(
    lambda c: (_toggle_decomp_visibility(), _run_decomposition()),
    names='value')
decomp_n_slider.observe(
    lambda c: _run_decomposition() if decomp_use_cb.value else None,
    names='value')
decomp_normalize_cb.observe(
    lambda c: _run_decomposition() if decomp_use_cb.value else None,
    names='value')


# ==============================================================================
# SAVE
# ==============================================================================

def on_save_plot(btn):
    if S['fig'] is None:
        save_html.value = '<i>Nothing to save yet.</i>'; return
    base = (os.path.splitext(S['files'][0]['name'])[0]
            if len(S['files']) == 1 else 'spectra_waterfall')
    out = f'{base}_fit.png'
    S['fig'].savefig(out, dpi=200, bbox_inches='tight')
    save_html.value = f'<b style="color:#080">Plot saved \u2192 <code>{out}</code></b>'


def on_save_data(btn):
    files = S['files']
    if not files:
        save_html.value = '<i>Nothing to save yet.</i>'; return
    visible_idxs = [i for i, cb in enumerate(files_box.children) if cb.value]
    visible = [files[i] for i in visible_idxs] or files

    if len(visible) == 1:
        fd = visible[0]
        if fd.get('x_trim') is None:
            save_html.value = '<i>Nothing to save yet.</i>'; return
        base = os.path.splitext(fd['name'])[0] or 'spectrum'
        out = f'{base}_fit.txt'
        ap = lam_slider.value if bl_algo.value == 'arPLS' else poly_deg.value
        save_fit_results(out, fd['x_trim'], fd['y_raw_trim'], fd['bl_trim'],
                         fd['y_sub'], fd['fit_curve'], fd['residual'],
                         fd.get('popt'), profile_rb.value,
                         bl_algo.value, ap, fd['name'])
        save_html.value = f'<b style="color:#080">Data saved \u2192 <code>{out}</code></b>'
    else:
        out = f'spectra_results_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
        save_results_table(out, visible, profile_rb.value, S['n_peaks'])
        save_html.value = (f'<b style="color:#080">Results table saved \u2192 '
                           f'<code>{out}</code></b> ({len(visible)} spectra)')


def on_save_summary(btn):
    if S['fig2'] is None:
        summary_save_html.value = '<i>Nothing to save yet.</i>'; return
    out = f'summary_pos_vs_fwhm_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.png'
    S['fig2'].savefig(out, dpi=200, bbox_inches='tight')
    summary_save_html.value = f'<b style="color:#080">Saved \u2192 <code>{out}</code></b>'


save_plot_btn.on_click(on_save_plot)
save_data_btn.on_click(on_save_data)
save_summary_btn.on_click(on_save_summary)


def on_save_decomp(btn):
    info = S.get('decomp')
    if info is None:
        decomp_save_html.value = '<i>Nothing to save yet.</i>'
        return
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    em_out = f'endmembers_{ts}.txt'
    wt_out = f'weights_{ts}.txt'
    x_common = info['x_common']; H = info['H']; W = info['W']
    names = info['names']
    # End-members: column 0 = wavenumber, then one column per component.
    em_data = np.column_stack([x_common, H.T])
    em_cols = ['x'] + [f'Component_{k+1}' for k in range(H.shape[0])]
    with open(em_out, 'w') as f:
        f.write(f'# End-member spectra (NMF, {H.shape[0]} components, '
                f'R\u00b2={info["r2"]:.4f}, '
                f'normalise={info["normalize"]})\n')
        f.write('# ' + '\t'.join(em_cols) + '\n')
        np.savetxt(f, em_data, fmt='%.6g', delimiter='\t')
    # Weights: rows = spectra, columns = components
    wt_cols = ['spectrum'] + [f'Component_{k+1}' for k in range(W.shape[1])]
    with open(wt_out, 'w') as f:
        f.write(f'# Component weights per spectrum (NMF)\n')
        f.write('# ' + '\t'.join(wt_cols) + '\n')
        for i, nm in enumerate(names):
            f.write(f'{nm}\t' + '\t'.join(f'{x:.6g}' for x in W[i]) + '\n')
    decomp_save_html.value = (
        f'<b style="color:#080">Saved \u2192 <code>{em_out}</code> and '
        f'<code>{wt_out}</code></b>')

decomp_save_btn.on_click(on_save_decomp)


# ==============================================================================
# LAYOUT
# ==============================================================================

hdr = lambda t: widgets.HTML(
    f'<h4 style="margin:8px 0 4px; border-bottom:1px solid #ccc">{t}</h4>')

app = widgets.VBox([
    hdr('1 \u2003 Load spectra'),
    widgets.HBox([file_upload, widgets.HTML('&nbsp; or &nbsp;'),
                  file_path_input, load_btn]),
    widgets.HBox([auto_cb, skip_input, maxr_input]),
    status_html,
    widgets.HBox([files_box,
                  widgets.VBox([tick_all_btn, untick_all_btn])]),
    ref_dropdown,

    hdr('2 \u2003 Wavenumber calibration'),
    widgets.HBox([cal_use_cb]),
    cal_body,

    hdr('3 \u2003 Background subtraction'),
    widgets.HBox([bl_algo,
                  widgets.VBox([lam_slider, poly_deg, trim_slider])]),

    hdr('4 \u2003 Peak fitting'),
    widgets.HBox([profile_rb,
                  widgets.VBox([widgets.HBox([ht_slider, peak_count_html]),
                                width_slider,
                                widgets.HBox([range_slider, range_reset_btn]),
                                offset_slider]),
                  widgets.VBox([widgets.HTML(
                      '<b>Manual peaks</b>'
                      '<div style="font-size:11px;color:#666">add centres for '
                      'shoulders / blends the<br>auto-detector misses</div>'),
                      manual_peaks_input, residual_auto_cb,
                      support_gate_cb],
                      layout=widgets.Layout(margin='0 0 0 16px'))]),

    # Outputs in vertical order so peak x-positions read straight down the page:
    #   explorer (one spectrum, raw + baseline + model)
    #   plot_out (waterfall + residual across all visible spectra)
    #   params_out (results table)
    explorer_out, plot_out, params_out,

    hdr('5 \u2003 Summary plot'),
    widgets.HBox([summary_x_dd, summary_y_dd]),
    summary_plot_out,
    peak_select_box,
    widgets.HBox([save_summary_btn, summary_save_html]),

    hdr('6 \u2003 Spectral decomposition'),
    decomp_intro_html,
    widgets.HBox([decomp_use_cb]),
    decomp_body,

    hdr('7 \u2003 Export'),
    widgets.HBox([save_plot_btn, save_data_btn, save_html]),
])

display(app)

## Future developments

Notes on known limitations / things to revisit, kept here for reference.

### The "continuous peak" limitation of spectral decomposition (Section 6)

NMF — and all linear matrix factorisation methods including MCR-ALS, PCA, and ICA — assume that the dataset's variance can be expressed as **linear mixtures of a small fixed set of pure spectra**. This assumption breaks down when a real peak in the dataset *shifts position continuously* across spectra (e.g. calcite's ν₁ peak drifting with Mg substitution, stress, temperature, hydration, etc.).

The mathematical reason: a linear combination `α·H₁ + β·H₂` of two pure spectra with peaks at positions p₁ and p₂ produces a *two-peak feature* (a smaller copy of each peak side by side), **not** a single peak smoothly interpolated between p₁ and p₂. So when the data really contains one peak that slides between p₁ and p₂, NMF discretises that continuum into N separate components, splitting one physical phase across two or more end-members. A continuum of shifted spectra has effectively *infinite rank* in the linear-mixing model, so any finite-N decomposition is forced to do something wrong.

**How to spot it in the notebook:** look at the parameter table beneath the abundance bar chart. If two components share essentially the same peak centres and FWHMs (just with different amplitudes), they are a degenerate decomposition of one physical phase — not two phases.

#### Approaches worth considering

In order of pragmatism for this notebook:

1. **Trust the parametric peak fit (Section 4) instead.** When peaks shift continuously, the summary scatter (Section 5) — especially with the new spectrum-index axis option — is the natural representation: you see the trajectory of centre, FWHM, etc. across the dataset directly, without forcing a small number of discrete "phases".

2. **Two-step: parametric peak removal, then NMF on the residual.** Use the per-spectrum peak fit to subtract the shifting Raman peaks (which are well-handled parametrically), then run NMF on the residual `Y_i − M_i`. The residual contains only what isn't a parametric peak — broad humps, amorphous backgrounds, etc. — which usually *is* a discrete mixture and is decomposed cleanly. Moderate-effort to wire in as an alternative pipeline (e.g. a "Decompose residual instead of raw spectra" checkbox in Section 6).

3. **PCA as a diagnostic.** PC2 of a dataset with one continuously shifting peak typically looks like the *derivative* of that peak (positive lobe + negative lobe). PCA loadings can therefore *flag* a continuous shift even though they aren't physically interpretable end-members. Could be exposed as a sibling toggle to NMF for diagnosis.

4. **Shift-tolerant variants of MCR.** Research-grade methods like ShiftNMF (Mørup et al.), MCR-ALS with peak-shift correction, or Bayesian factor models with parametric peak priors that allow per-spectrum shifts of each component. Available in libraries like `nimfa`, `pymcr`, and various MATLAB toolboxes — not in scikit-learn. Would solve the problem cleanly for hybrid datasets (some discrete mixing + some continuous shifts) but require more setup and have more knobs to tune.

For the typical Diamond/B17 calcite-plus-amorphous workflow, option (1) combined with the existing baseline-subtraction integrated area is probably enough; option (2) is the obvious next addition if the residual NMF gives noticeably cleaner end-members in practice.